# Recursive Language Model ([RLM](https://github.com/alexzhang13/rlm)) with [Pydantic Monty](https://github.com/pydantic/monty) sandbox

RLM's treat context as a variable in a python sandbox. This allows the LLM to programmatically examine, decompose, and recursively call itself over its input. Context is controlled through the REPL (read evaluate print loop) as the model only 'sees' standard output, e.g., print statements, error messages. 

This notebook demonstrates an RLM implementation built using `pydantic-monty` code sandbox, which is a minimal Python interpreter that supports a small but useful subset of Python code. For context we'll use a publicly available [International Building Code](https://core-docs.s3.amazonaws.com/documents/asset/uploaded_file/3836/Lennox/3042180/2021_International_Building_Code.pdf).

In [1]:
import os
import sys
from IPython.display import Markdown, display

from pydantic import BaseModel, Field

In [2]:
parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)

## Set up the RLM

The RLM has three main tools:
- `SAVE` for persisting objects
- `SUBMIT` to exit the loop and return a final answer
- `query_llm` for spawning sub-llm

The loop continues until `SUBMIT` is called or `max_iterations` is reached, in which case the model is asked to review the full trace and answer as best as it can.

In [3]:
from src.rlm import MontyRLM

In [4]:
import nest_asyncio
from openai import AsyncOpenAI
from pydantic_ai.models import Model
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

nest_asyncio.apply()

def get_openai_model(model_name: str) -> Model:
    """Get an OpenAI model from a model name."""

    client = AsyncOpenAI()
    model = OpenAIChatModel(model_name, provider=OpenAIProvider(openai_client=client))
    return model



class Answer(BaseModel):
    reasoning: str = Field(description="The reasoning for the answer")
    answer: str = Field(description="The answer to the question")
    

rlm = MontyRLM(
    output_type=Answer,
    model=get_openai_model("gpt-5.2"), # <- main model
    sub_model=get_openai_model("gpt-4.1-mini"), # <- cheaper sub-llm
    max_iterations=20,
    max_llm_calls=50,
)

In [5]:
import pymupdf

doc = pymupdf.open('../data/2021_International_Building_Code.pdf')

text = ""
pages = []

for page in doc:
    text += page.get_text()
    pages.append(page.get_text())
    
    
print(f"Total pages: {len(pages)}")

Total pages: 833


In [6]:
import tiktoken

def count_tokens_tiktoken(text: str) -> int:
    return len(tiktoken.encoding_for_model("gpt-5").encode(text))


tokens = count_tokens_tiktoken(text)

print(f"Total tokens: {tokens:,}")

Total tokens: 1,037,619


* Example page

In [7]:
print(pages[261])

2021 INTERNATIONAL BUILDING CODE®
CHAPTER 9
 FIRE PROTECTION AND LIFE SAFETY SYSTEMS
User note:
About this chapter: Chapter 9 prescribes the minimum requirements for active fire protection equipment systems to perform the functions
of detecting a fire, alerting the occupants or fire department of a fire emergency, mass notification, gas detection, controlling smoke and
controlling or extinguishing the fire. Generally, the requirements are based on the occupancy, the height and the area of the building,
because these are the factors that most affect fire-fighting capabilities and the relative hazard of a specific building or portion thereof. This
chapter parallels and is substantially duplicated in Chapter 9 of the International Fire Code®.
SECTION 901
GENERAL
901.1 Scope. The provisions of this chapter shall specify
where fire protection and life safety systems are required and
shall apply to the design, installation and operation of fire
protection and life safety systems.
901.2 Fire 

## Run RLM query

In [ ]:
result = await rlm.run(
    query="what do the building codes say about composition shingle installation?",
    context=pages,
)

## Inspect REPL History

In [9]:
display(Markdown(result.render_trajectory()))

### Iteration 1

> **Reasoning:** Need to explore provided query/context structure and contents before answering.

```python
import json, re
print(type(query), type(context))
print('query:', query)
print('context_len:', len(context))
print('context_item_type_counts:', {type(x).__name__: sum(isinstance(i, type(x)) for i in context) for x in context[:10]})
print('first_item_preview:', repr(context[0])[:400])
```

**Output:**
```
[Error] TypeError: unsupported operand type(s) for +: 'int' and 'bool'
```

---

### Iteration 2

> **Reasoning:** The previous exploration code errored due to summing booleans with an int seed incorrectly when building type counts. First, inspect query/context types, lengths, and a few previews safely.

```python
import re, json
print('query_type:', type(query).__name__)
print('context_type:', type(context).__name__)
print('query:', query)

# context may be list-like
try:
    n = len(context)
except Exception as e:
    n = None
    print('len(context) failed:', e)
print('context_len:', n)

# show first few items safely
if isinstance(context, (list, tuple)):
    for i in range(min(5, len(context))):
        item = context[i]
        s = repr(item)
        print(f'context[{i}] type:', type(item).__name__, 'preview:', s[:300])
else:
    print('context preview:', repr(context)[:500])
```

**Output:**
```
query_type: str
context_type: list
query: what do the building codes say about composition shingle installation?
context_len: 833
context[0] type: str preview: '101167924\n'
context[1] type: str preview: 'IBC\n®\nINTERNATIONAL  \nBUILDING  \nCODE\n® \nA Member of the International Code Family®\nCopyright © 2020 ICC.  ALL RIGHTS RESERVED. Accessed by Nathan Vander Plaats (nvp@cityoflennoxsd.com), (-) Order Number #101167924 on Oct 14, 2021 06:19 AM (PDT) pursuant to License Agreement\nwith ICC.  No f
context[2] type: str preview: '2021 International Building Code\nDate of First Publication: October 23, 2020\nFirst Printing: October 2020\nISBN: 978-1-60983-955-0 (soft-cover edition)\nISBN: 978-1-60983-956-7 (loose-leaf edition)\nCOPYRIGHT  2020\nby\nINTERNATIONAL CODE COUNCIL, INC.\nALL RIGHTS RESERVED. This 2021 Internatio
context[3] type: str preview: '2021 INTERNATIONAL BUILDING CODE®\niii\nPREFACE\nIntroduction\n The International Building Code® (IBC®) establishes minimum requirements for building systems using\nprescriptive and performance-related provisions. It is founded on broad-based principles that make\npossible the use of new materials 
context[4] type: str preview: '2021 INTERNATIONAL BUILDING CODE®\niv\nMaintenance\nThe IBC is kept up to date through the review of proposed changes submitted by code enforcement\nofficials, industry representatives, design professionals and other interested parties. Proposed\nchanges are carefully considered through an open cod

```

---

### Iteration 3

> **Reasoning:** We need find relevant building code sections about composition (asphalt) shingles installation in the provided IBC context list. We'll search within context strings for keywords (asphalt, shingle, roof covering, R905, IRC, IBC Chapter 15) then use llm_query if needed. First do python search to locate excerpts.

```python
import re
# Join context into one big text for searching
text = "".join(context)
print('total_chars', len(text))
# Find occurrences of key terms
terms = ['asphalt', 'shingle', 'composition', 'R905', 'roof covering', '1507', '1507.2', 'ASTM D3462', 'underlayment', 'ice barrier', 'fasten', 'nail', 'slope']
for t in terms:
    m = re.search(re.escape(t), text, re.IGNORECASE)
    print(t, 'found' if m else 'not_found', ('at '+str(m.start()) if m else ''))

# Extract nearby snippets around first few shingle mentions
matches = list(re.finditer(r'shingl', text, re.IGNORECASE))
print('shingle_matches', len(matches))
for m in matches[:5]:
    start = max(0, m.start()-500)
    end = min(len(text), m.end()+800)
    snippet = text[start:end]
    print('\n--- snippet ---\n', snippet)
```

**Output:**
```
total_chars 3736848
asphalt found at 272461
shingle found at 48522
composition found at 242309
R905 not_found 
roof covering found at 48620
1507 found at 91297
1507.2 found at 1999895
ASTM D3462 found at 2007172
underlayment found at 251722
ice barrier found at 1999015
fasten found at 166154
nail found at 252159
slope found at 99826
shingle_matches 148

--- snippet ---
 er. 
Chapter 15 Roof Assemblies and Rooftop Structures
Chapter 15 provides standards for both roof assemblies and structures that sit on top of the roofs of
buildings. The criteria address roof construction and covering, including the weather-protective bar-
rier at the roof and, in most circumstances, a fire-resistant barrier. The chapter is prescriptive in
nature and is based on decades of experience with various traditional materials, but it also addresses
newer products such as photovoltaic shingles. These prescriptive rules are very important for satisfy-
ing performance of one type of roof covering or another. Section 1511 addresses rooftop structures,
including penthouses, tanks, towers and spires. Rooftop penthouses larger than prescribed in this
chapter must be treated as a story under Chapter 5. 
Chapter 16 Structural Design
Chapter 16 prescribes minimum structural loading requirements for use in the design and construc-
tion of buildings and structural components. It includes minimum design loads, assignment of risk cat-
egories and permitted design methodologies. Standards are provided for minimum design loads (live,
dead, snow, wind, rain, flood, ice and earthquake as well as the required load combinations). The
application of these loads and adherence to the serviceabilit

--- snippet ---
 AND CEILING FINISH. The
exposed interior surfaces of buildings, including but not
limited to: fixed or movable walls and partitions; toilet room
privacy partitions; columns; ceilings; and interior wainscot-
ing, paneling or other finish applied structurally or for
decoration, acoustical correction, surface insulation, struc-
tural fire resistance or similar purposes, but not including
trim.
[BS] INTERLAYMENT. A layer of felt or nonbituminous
saturated felt not less than 18 inches (457 mm) wide, shingled
between each course of a wood-shake roof covering.
[BS] INTERMODAL SHIPPING CONTAINER. A six-
sided steel unit originally constructed as a general cargo
container used for the transport of goods and materials.
[BF] INTUMESCENT FIRE-RESISTANT COATINGS.
Thin film liquid mixture applied to substrates by brush, roller,
spray or trowel which expands into a protective foamed layer
to provide fire-resistant protection of the substrates when
exposed to flame or intense heat.
[BS] JOINT. The opening in or between adjacent assemblies
that is created due to building tolerances, or is designed to
allow independent movement of the building in any plane
caused by thermal, seismic, wind or any other loading.
[A] JURISDICTION. The governmental unit that has
adopted this code.
[BF] L RATING. The air lea

--- snippet ---
 EEMENT, AND SUBJECT TO CIVIL AND CRIMINAL PENALTIES THEREUNDER.
101167924
DEFINITIONS
2021 INTERNATIONAL BUILDING CODE®
[BF] METAL COMPOSITE MATERIAL (MCM)
SYSTEM. An exterior wall covering fabricated using MCM
in a specific assembly including joints, seams, attachments,
substrate, framing and other details as appropriate to a partic-
ular design.
[BS] METAL ROOF PANEL. An interlocking metal sheet
having a minimum installed weather exposure of 3 square
feet (0.279 m2) per sheet.
[BS] METAL ROOF SHINGLE. An interlocking metal
sheet having an installed weather exposure less than 3 square
feet (0.279 m2) per sheet.
[BG] MEZZANINE. An intermediate level or levels
between the floor and ceiling of any story and in accordance
with Section 505.
[BS] MICROPILE. A micropile is a bored, grouted-in-place
deep foundation element that develops its load-carrying
capacity by means of a bond zone in soil, bedrock or a combi-
nation of soil and bedrock.
[BF] MINERAL BOARD. A rigid felted thermal insulation
board consisting of either felted mineral fiber or cellular
beads of expanded aggregate formed into flat rectangular
units.
[BF] MINERAL FIBER. Insulation composed principally
of fibers manufactured from rock, slag or glass, with or with-
out binders.
[BF] MINERAL WOOL. Synthetic vitreous fiber insula

--- snippet ---
 omplete, environ-
mentally protected unit consisting of solar cells, optics and
other components, exclusive of tracker, designed to generate
DC power when exposed to sunlight.
[BS] PHOTOVOLTAIC PANEL. A collection of modules
mechanically fastened together, wired and designed to
provide a field-installable unit.
[BS] PHOTOVOLTAIC PANEL SYSTEM. A system that
incorporates discrete photovoltaic panels, that converts solar
radiation into electricity, including rack support systems.
[BS] PHOTOVOLTAIC SHINGLES. A roof covering
resembling shingles that incorporates photovoltaic modules.
[F] PHYSICAL HAZARD. A chemical for which there is
evidence that i
... [truncated]
```

---

### Iteration 4

> **Reasoning:** We need to answer what building codes say about composition (asphalt) shingle installation. Context is 2021 IBC text; relevant provisions likely in Chapter 15, Section 1507.2 Asphalt shingles and underlayment/ice barrier. Next step: locate exact excerpts around '1507.2' and 'ASTM D3462' and extract requirements. We'll use Python to find and print that section from the joined text for accurate quoting.

```python
import re
text = ''.join(context)
# Find start near Section 1507.2
m = re.search(r'1507\.2\s+Asphalt shingles', text)
print('found 1507.2 header:', bool(m), 'pos', m.start() if m else None)
# fallback search
if not m:
    m = re.search(r'1507\.2', text)
    print('found 1507.2 generic:', bool(m), 'pos', m.start() if m else None)

if m:
    start = max(0, m.start()-2000)
    end = min(len(text), m.start()+20000)
    excerpt = text[start:end]
    print(excerpt)

# Also find ice barrier clause
mi = re.search(r'ice barrier', text, re.IGNORECASE)
print('ice barrier pos', mi.start() if mi else None)
if mi:
    s=max(0, mi.start()-1500); e=min(len(text), mi.start()+4000)
    print('\n---ice barrier excerpt---\n', text[s:e])
```

**Output:**
```
found 1507.2 header: True pos 2006465
ll be one layer 
applied as follows: Underlayment shall 
be applied shingle fashion, parallel to 
and starting from the eave and lapped 
4 inches. End laps shall be 4 inches 
and shall be offset by 6 feet.
Metal roof 
shingles
1507.5
Mineral-
surfaced roll 
roofing
1507.6
Slate shingles
1507.7
Wood shingles
1507.8
Wood shakes
1507.9
Photovoltaic 
shingles
1507.16
For roof slopes from 3 units vertical in 12 units horizontal (3:12), up 
to 4 units vertical in 12 units horizontal (4:12), underlayment shall be 
two layers applied as follows: Apply a 19-inch strip of underlayment 
felt parallel to and starting at the eaves. Starting at the eave, apply 36-
inch-wide sheets of underlayment, overlapping successive sheets 19 
inches. End laps shall be 4 inches and shall be offset by 6 feet. Distor-
tions in the underlayment shall not interfere with the ability of the 
shingles to seal.
For roof slopes of 4 units vertical in 12 units horizontal (4:12) or 
greater, underlayment shall be one layer applied as follows: Under-
layment shall be applied shingle fashion, parallel to and starting from 
the eave and lapped 2 inches. Distortions in the underlayment shall 
not interfere with the ability of the shingles to seal. End laps shall be 
4 inches and shall be offset by 6 feet.
Same as Maximum Basic Design 
Wind Speed, V < 140 mph except all 
laps shall be not less than 4 inches
Copyright © 2020 ICC.  ALL RIGHTS RESERVED. Accessed by Nathan Vander Plaats (nvp@cityoflennoxsd.com), (-) Order Number #101167924 on Oct 14, 2021 06:19 AM (PDT) pursuant to License Agreement
with ICC.  No further reproduction, no further reproductions by any third party, or distribution authorized.  Single user only, copying and networking prohibited. ANY UNAUTHORIZED REPRODUCTION OR
DISTRIBUTION IS A VIOLATION OF THE FEDERAL COPYRIGHT ACT AND THE LICENSE AGREEMENT, AND SUBJECT TO CIVIL AND CRIMINAL PENALTIES THEREUNDER.
101167924
 ROOF ASSEMBLIES AND ROOFTOP STRUCTURES
2021 INTERNATIONAL BUILDING CODE®
1507.2 Asphalt shingles. The installation of asphalt shingles
shall comply with the provisions of this section.
1507.2.1 Deck requirements. Asphalt shingles shall be
fastened to solidly sheathed decks.
1507.2.2 Slope. Asphalt shingles shall only be used on
roof slopes of 2 units vertical in 12 units horizontal (17-
percent slope) or greater. For roof slopes from 2 units
vertical in 12 units horizontal (17-percent slope) up to 4
units vertical in 12 units horizontal (33-percent slope),
double underlayment application is required in accor-
dance with Section 1507.2.8.
1507.2.3 Underlayment. Underlayment shall comply
with Section 1507.1.1.
1507.2.4 Asphalt shingles. Asphalt shingles shall comply
with ASTM D3462.
1507.2.5 Fasteners. Fasteners for asphalt shingles shall
be galvanized, stainless steel, aluminum or copper roof-
ing nails, minimum 12-gage [0.105 inch (2.67 mm)]
shank with a minimum 3/8-inch-diameter (9.5 mm) head,
of a length to penetrate through the roofing materials and
not less than 3/4 inch (19.1 mm) into the roof sheathing.
Where the roof sheathing is less than 3/4 inch (19.1 mm)
thick, the nails shall penetrate through the sheathing.
Fasteners shall comply with ASTM F1667.
1507.2.6 Attachment. Asphalt shingles shall have the
minimum number of fasteners required by the manufac-
turer, but not less than four fasteners per strip shingle or
two fasteners per individual shingle. Where the roof slope
exceeds 21 units vertical in 12 units horizontal (21:12),
shingles shall be installed as required by the manufacturer.
1507.2.7 Ice barrier. Where required, ice barriers shall
comply with Section 1507.1.2.
1507.2.8 Flashings. Flashing for asphalt shingles shall
comply with this section. Flashing shall be applied in
accordance with this section and the asphalt shingle
manufacturer’s printed instructions.
1507.2.8.1 Base and cap flashing. Base and cap flash-
ing shall be installed in accordance with the
manufacturer’s instructions. Base flashing shall be of
either corrosion-resistant metal of minimum nominal
0.019-inch (0.483 mm) thickness or mineral-surfaced
roll roofing weighing not less than 77 pounds per 100
square feet (3.76 kg/m2). Cap flashing shall be corro-
sion-resistant metal of minimum nominal 0.019-inch
(0.483 mm) thickness.
1507.2.8.2 Valleys. Valley linings shall be installed in
accordance with the manufacturer’s instructions before
applying shingles. Valley linings of the following
types shall be permitted:
1.
For open valleys (valley lining exposed) lined
with metal, the valley lining shall be not less
than 24 inches (610 mm) wide and of any of
15-7
TABLE 1507.1.1(3)
UNDERLAYMENT ATTACHMENT
For SI: 1 inch = 25.4 mm; 1 mile per hour = 0.447 m/s.
ROOF COVERING
SECTION
MAXIMUM BASIC 
DESIGN WIND SPEED, 
V < 140 MPH
MAXIMUM BASIC DESIGN WIND SPEED, V ≥ 140 MPH
Asphalt shingles
1507.2
Fastened sufficiently 
to hold in place
The underlayment shall be attached with corrosion-resistant fasten-
ers
... [truncated]
```

---

### Iteration 5

> **Reasoning:** We have extracted the key IBC 2021 provisions for asphalt (composition) shingles from context. Now craft a concise answer summarizing installation requirements: deck, slope, underlayment types/application incl high-wind laps, ice barrier, material standard, fasteners specs, attachment counts, flashing/valleys/drip edge. Then submit via SUBMIT with reasoning and answer.

```python
import re
text=''.join(context)
# Extract key sections precisely for reference
patterns = {
 '1507.2':'1507\.2 Asphalt shingles',
 '1507.1.2':'1507\.1\.2 Ice barriers',
 'Table1507.1.1(2)':'TABLE 1507\.1\.1\(2\)\s+UNDERLAYMENT APPLICATION',
 'Table1507.1.1(1)':'TABLE 1507\.1\.1\(1\)\s+UNDERLAYMENT TYPES',
}
excerpts={}
for k,p in patterns.items():
    m=re.search(p, text)
    if m:
        excerpts[k]=text[m.start():m.start()+2500]
print('found excerpts:', list(excerpts))
print(excerpts['1507.2'][:1200])
print('---')
print(excerpts['1507.1.2'][:900])
```

**Output:**
```
found excerpts: ['1507.2', '1507.1.2', 'Table1507.1.1(2)', 'Table1507.1.1(1)']
1507.2 Asphalt shingles. The installation of asphalt shingles
shall comply with the provisions of this section.
1507.2.1 Deck requirements. Asphalt shingles shall be
fastened to solidly sheathed decks.
1507.2.2 Slope. Asphalt shingles shall only be used on
roof slopes of 2 units vertical in 12 units horizontal (17-
percent slope) or greater. For roof slopes from 2 units
vertical in 12 units horizontal (17-percent slope) up to 4
units vertical in 12 units horizontal (33-percent slope),
double underlayment application is required in accor-
dance with Section 1507.2.8.
1507.2.3 Underlayment. Underlayment shall comply
with Section 1507.1.1.
1507.2.4 Asphalt shingles. Asphalt shingles shall comply
with ASTM D3462.
1507.2.5 Fasteners. Fasteners for asphalt shingles shall
be galvanized, stainless steel, aluminum or copper roof-
ing nails, minimum 12-gage [0.105 inch (2.67 mm)]
shank with a minimum 3/8-inch-diameter (9.5 mm) head,
of a length to penetrate through the roofing materials and
not less than 3/4 inch (19.1 mm) into the roof sheathing.
Where the roof sheathing is less than 3/4 inch (19.1 mm)
thick, the nails shall penetrate through the sheathing.
Fasteners shall comply with ASTM 
---
1507.1.2 Ice barriers. In areas where there has been a
history of ice forming along the eaves causing a backup
of water, an ice barrier shall be installed for asphalt shin-
gles, metal roof shingles, mineral-surfaced roll roofing,
slate and slate-type shingles, wood shingles, and wood
shakes. The ice barrier shall consist of not less than two
layers of underlayment cemented together, or a self-
adhering polymer modified bitumen sheet shall be used in
place of normal underlayment and extend from the lowest
edges of all roof surfaces to a point not less than 24 inches
(610 mm) inside the exterior wall line of the building.
Exception: Detached accessory structures that do not
contain conditioned floor area.

15-5
TABLE 1507.1.1(1)
UNDERLAYMENT TYPES
ROOF COVERING
SECTION
MAXIMUM BASIC DESIGN 
WIND SPEED, V < 140 MPH
MAXIMUM BASIC DESIGN 
WIND SPEED, V ≥ 140 MPH
Asphalt shingles
1507.2
ASTM

```

---

### Iteration 6

> **Reasoning:** We already located the relevant 2021 IBC Chapter 15 Section 1507.2 and related underlayment/ice barrier provisions in the provided context. Now we should craft the final response summarizing what the code requires for composition (asphalt) shingle installation, citing the key section numbers and main installation criteria (deck, slope limits, underlayment types/application, ice barrier, fasteners, attachment, flashing/valleys/drip edge, wind-speed table notes).

```python
import re
text=''.join(context)
# pull a bit more of 1507.2.5-1507.2.8.3 for fastening/attachment/drip edge details
m=re.search(r'1507\.2 Asphalt shingles\.', text)
excerpt=text[m.start():m.start()+9000]
print(excerpt)
```

**Output:**
```
1507.2 Asphalt shingles. The installation of asphalt shingles
shall comply with the provisions of this section.
1507.2.1 Deck requirements. Asphalt shingles shall be
fastened to solidly sheathed decks.
1507.2.2 Slope. Asphalt shingles shall only be used on
roof slopes of 2 units vertical in 12 units horizontal (17-
percent slope) or greater. For roof slopes from 2 units
vertical in 12 units horizontal (17-percent slope) up to 4
units vertical in 12 units horizontal (33-percent slope),
double underlayment application is required in accor-
dance with Section 1507.2.8.
1507.2.3 Underlayment. Underlayment shall comply
with Section 1507.1.1.
1507.2.4 Asphalt shingles. Asphalt shingles shall comply
with ASTM D3462.
1507.2.5 Fasteners. Fasteners for asphalt shingles shall
be galvanized, stainless steel, aluminum or copper roof-
ing nails, minimum 12-gage [0.105 inch (2.67 mm)]
shank with a minimum 3/8-inch-diameter (9.5 mm) head,
of a length to penetrate through the roofing materials and
not less than 3/4 inch (19.1 mm) into the roof sheathing.
Where the roof sheathing is less than 3/4 inch (19.1 mm)
thick, the nails shall penetrate through the sheathing.
Fasteners shall comply with ASTM F1667.
1507.2.6 Attachment. Asphalt shingles shall have the
minimum number of fasteners required by the manufac-
turer, but not less than four fasteners per strip shingle or
two fasteners per individual shingle. Where the roof slope
exceeds 21 units vertical in 12 units horizontal (21:12),
shingles shall be installed as required by the manufacturer.
1507.2.7 Ice barrier. Where required, ice barriers shall
comply with Section 1507.1.2.
1507.2.8 Flashings. Flashing for asphalt shingles shall
comply with this section. Flashing shall be applied in
accordance with this section and the asphalt shingle
manufacturer’s printed instructions.
1507.2.8.1 Base and cap flashing. Base and cap flash-
ing shall be installed in accordance with the
manufacturer’s instructions. Base flashing shall be of
either corrosion-resistant metal of minimum nominal
0.019-inch (0.483 mm) thickness or mineral-surfaced
roll roofing weighing not less than 77 pounds per 100
square feet (3.76 kg/m2). Cap flashing shall be corro-
sion-resistant metal of minimum nominal 0.019-inch
(0.483 mm) thickness.
1507.2.8.2 Valleys. Valley linings shall be installed in
accordance with the manufacturer’s instructions before
applying shingles. Valley linings of the following
types shall be permitted:
1.
For open valleys (valley lining exposed) lined
with metal, the valley lining shall be not less
than 24 inches (610 mm) wide and of any of
15-7
TABLE 1507.1.1(3)
UNDERLAYMENT ATTACHMENT
For SI: 1 inch = 25.4 mm; 1 mile per hour = 0.447 m/s.
ROOF COVERING
SECTION
MAXIMUM BASIC 
DESIGN WIND SPEED, 
V < 140 MPH
MAXIMUM BASIC DESIGN WIND SPEED, V ≥ 140 MPH
Asphalt shingles
1507.2
Fastened sufficiently 
to hold in place
The underlayment shall be attached with corrosion-resistant fasten-
ers in a grid pattern of 12 inches between side laps with a 6-inch 
spacing at side and end laps. Underlayment shall be attached using 
metal or plastic cap nails or cap staples with a nominal cap diameter 
of not less than 1 inch. Metal caps shall have a thickness of not less 
than 32-gage (0.0134 inch) sheet metal. Power-driven metal caps 
shall have a minimum thickness of 0.010 inch. Minimum thickness 
of the outside edge of plastic caps shall be 0.035 inch. The cap nail 
shank shall be not less than 0.083 inch for ring shank cap nails and 
0.091 inch for smooth shank cap nails. Staples shall be not less than 
21 gage (0.032 inch). The cap nail shank and cap staple legs shall 
have a length sufficient to penetrate through the roof sheathing or 
not less than 3/4 inch into the roof sheathing.
Clay and concrete tile
1507.3
Photovoltaic shingles
1507.16
Metal roof panels
1507.4
Manufacturer’s 
installation 
instructions
The underlayment shall be attached with corrosion-resistant fasten-
ers in a grid pattern of 12 inches between side laps with a 6-inch 
spacing at side and end laps. Underlayment shall be attached using 
metal or plastic cap nails or cap staples with a nominal cap diameter 
of not less than 1 inch. Metal caps shall have a thickness of not less 
than 32-gage sheet metal. Power-driven metal caps shall have a 
minimum thickness of 0.010 inch. Minimum thickness of the 
outside edge of plastic caps shall be 0.035 inch. The cap nail shank 
shall be not less than 0.083 inch for ring shank cap nails and 0.091 
inch for smooth shank cap nails. Staples shall be not less than 21 
gage. The cap nail shank and cap staple legs shall have a length 
sufficient to penetrate through the roof sheathing or not less than 3/4 
inch into the roof sheathing.
Metal roof shingles
1507.5
Mineral-surfaced roll roofing
1507.6
Slate shingles
1507.7
Wood shingles
1507.8
Wood shakes
1507.9
Copyright © 2020 ICC.  ALL RIGHTS RESERVED. Accessed by Nathan Vander Plaats (nvp@cityoflennoxsd.com), (-) Order Number #101167924 on 
... [truncated]
```

---

### Iteration 7

> **Reasoning:** We have the relevant 2021 IBC Section 1507.2 (asphalt/composition shingles) and related Section 1507.1.1 tables (underlayment types/application/attachment) and Section 1507.1.2 (ice barriers). Provide a concise summary of code requirements with section references and key numeric criteria (deck, slope, underlayment layering and laps, fasteners, attachment count, ice barrier extent, drip edge, valleys/flashings).

```python
# No further computation needed; proceed to final response.
```

**Output:**
```
(no output - did you forget to print?)
```

---

### Iteration 8

> **Reasoning:** We have already extracted the relevant provisions from the provided 2021 International Building Code (IBC) context—primarily Section 1507.2 (Asphalt shingles) plus underlayment/ice-barrier requirements in Sections 1507.1.1 and 1507.1.2. Provide a concise, code-referenced summary addressing installation of composition/asphalt shingles: deck, slope limits, underlayment types and application (including higher-wind adjustments), ice barrier conditions/extent, shingle and fastener standards, minimum fastening, and key flashing/valley/drip-edge rules.

```python
# No further sandbox work required; ready to provide final summarized answer with IBC section references.
```

**Output:**
```
(no output - did you forget to print?)
```

---

### Iteration 9

> **Reasoning:** We already extracted the relevant 2021 IBC provisions from the provided context (Chapter 15, §§1507.2, 1507.1.1 tables, and 1507.1.2). Now provide a concise, code-referenced summary of what the code requires for composition/asphalt shingle installation (deck, slope, underlayment, ice barrier, shingle standard, fasteners, flashing/valleys/drip edge, and high-wind underlayment attachment).

```python
# Summarize key IBC requirements from extracted excerpts (already inspected in prior iterations)
# No further computation required.
print('Ready to submit summarized IBC 2021 requirements for asphalt/composition shingle installation.')
```

**Output:**
```
Ready to submit summarized IBC 2021 requirements for asphalt/composition shingle installation.

```

---

### Iteration 10

> **Reasoning:** Proceed to submit final response summarizing 2021 IBC (Chapter 15) requirements for asphalt/composition shingle installation, citing key sections and numeric criteria extracted from context.

```python
# Prepare final reasoning and answer, then submit.
reasoning = (
    "From the provided context (2021 IBC Chapter 15), composition/asphalt shingles are regulated under "
    "IBC §1507.2 with supporting requirements in §1507.1.1 (underlayment types/application/attachment) "
    "and §1507.1.2 (ice barriers). Summarize those provisions with key numeric installation criteria."
)
answer = (
    "From the 2021 International Building Code (IBC), Chapter 15 (Roof Assemblies and Rooftop Structures), "
    "‘composition’ shingles are addressed as **asphalt shingles** (IBC **§1507.2**), with underlayment and ice-barrier "
    "rules in **§1507.1.1** and **§1507.1.2**:\n\n"
    "### IBC §1507.2 — Asphalt (composition) shingles\n"
    "- **Deck**: Must be **fastened to solidly sheathed decks** (§1507.2.1).\n"
    "- **Minimum slope**: Allowed only on roofs **≥ 2:12** (§1507.2.2).\n"
    "  - For slopes **2:12 up to 4:12**, **double underlayment** is required (§1507.2.2; see underlayment application in §1507.1.1/Table 1507.1.1(2)).\n"
    "- **Shingle standard**: Shingles must comply with **ASTM D3462** (§1507.2.4).\n"
    "- **Fasteners (nails)** (§1507.2.5):\n"
    "  - Must be **galvanized, stainless steel, aluminum, or copper roofing nails**.\n"
    "  - **Minimum 12-gage** (0.105 in) shank and **minimum 3/8-in head**.\n"
    "  - Length must penetrate through roofing and **≥ 3/4 in into the roof sheathing** (or **through** the sheathing if sheathing is < 3/4 in).\n"
    "  - Fasteners must comply with **ASTM F1667**.\n"
    "- **Minimum attachment** (§1507.2.6):\n"
    "  - At least the manufacturer’s requirement, but **not less than 4 fasteners per strip shingle** or **2 per individual shingle**.\n"
    "  - For very steep roofs (**> 21:12**), install as required by the manufacturer.\n"
    "- **Ice barrier**: Where required, must comply with **§1507.1.2** (§1507.2.7).\n"
    "- **Flashings/valleys/drip edge** (§1507.2.8):\n"
    "  - Flashing must follow this section **and the manufacturer’s printed instructions**.\n"
    "  - **Base/cap flashing**: corrosion-resistant metal (nominal **0.019 in** thick) or qualifying mineral-surfaced roll roofing for base; cap flashing corrosion-resistant metal nominal **0.019 in** (§1507.2.8.1).\n"
    "  - **Valleys**: Valley linings installed before shingles; open/closed valley options are permitted, including specified roll-roofing/underlayment options (§1507.2.8.2).\n"
    "  - **Drip edge required** at eaves and rake edges (§1507.2.8.3):\n"
    "    - Lap adjacent pieces **≥ 2 in**.\n"
    "    - Vertical leg **≥ 1.5 in** and extends **≥ 1/4 in below** sheathing; extends back on roof **≥ 2 in**.\n"
    "    - Install **underlayment over drip edge at eaves**; install **drip edge over underlayment at rakes**.\n"
    "    - Fasten at **≤ 12 in o.c.**\n\n"
    "### Underlayment (IBC §1507.1.1)\n"
    "- **Underlayment required** for asphalt shingles must comply with §1507.1.1 (referenced by §1507.2.3).\n"
    "- **Underlayment types** are specified in **Table 1507.1.1(1)** (by wind speed category).\n"
    "- **Underlayment application** for asphalt shingles is in **Table 1507.1.1(2)**:\n"
    "  - **2:12 up to 4:12**: **two layers** applied with a **19-in starter strip** at eave; then 36-in sheets overlapping **19 in**; **end laps 4 in** and offset **6 ft**; underlayment must lay flat enough to not prevent shingles sealing.\n"
    "  - **≥ 4:12**: **one layer** applied shingle-fashion starting at eave; **lap 2 in**; **end laps 4 in** offset **6 ft**; must not interfere with shingle sealing.\n"
    "  - For higher wind category shown, same as low-wind but **all laps ≥ 4 in**.\n"
    "- **Underlayment attachment** for asphalt shingles in high-wind category is in **Table 1507.1.1(3)** (grid fastening with cap nails/staples and specified spacings/materials).\n\n"
    "### Ice barriers (IBC §1507.1.2)\n"
    "- Where there is a **history of ice forming along eaves causing backup of water**, an **ice barrier is required** for asphalt shingles (and other listed coverings) (§1507.1.2).\n"
    "- Must be **two layers of underlayment cemented together** *or* a **self-adhering polymer-modified bitumen sheet**.\n"
    "- Must extend from the roof’s lowest edges to a point **≥ 24 in inside the exterior wall line** (§1507.1.2).\n"
    "- Exception: **Detached accessory structures** without conditioned floor area.\n\n"
    "Note: The IBC repeatedly requires following the **manufacturer’s printed installation instructions** in addition to these minimum code provisions (notably for attachment on very steep slopes and for flashings/valleys)."
)
SUBMIT(reasoning=reasoning, answer=answer)
```

**Output:**
```
SUBMIT: reasoning='From the provided context (2021 IBC Chapter 15), composition/asphalt shingles are regulated under IBC §1507.2 with supporting requirements in §1507.1.1 (underlayment types/application/attachment) and §1507.1.2 (ice barriers). Summarize those provisions with key numeric installation criteria.' answer='From the 2021 International Building Code (IBC), Chapter 15 (Roof Assemblies and Rooftop Structures), ‘composition’ shingles are addressed as **asphalt shingles** (IBC **§1507.2**), with underlayment and ice-barrier rules in **§1507.1.1** and **§1507.1.2**:\n\n### IBC §1507.2 — Asphalt (composition) shingles\n- **Deck**: Must be **fastened to solidly sheathed decks** (§1507.2.1).\n- **Minimum slope**: Allowed only on roofs **≥ 2:12** (§1507.2.2).\n  - For slopes **2:12 up to 4:12**, **double underlayment** is required (§1507.2.2; see underlayment application in §1507.1.1/Table 1507.1.1(2)).\n- **Shingle standard**: Shingles must comply with **ASTM D3462** (§1507.2.4).\n- **Fasteners (nails)** (§1507.2.5):\n  - Must be **galvanized, stainless steel, aluminum, or copper roofing nails**.\n  - **Minimum 12-gage** (0.105 in) shank and **minimum 3/8-in head**.\n  - Length must penetrate through roofing and **≥ 3/4 in into the roof sheathing** (or **through** the sheathing if sheathing is < 3/4 in).\n  - Fasteners must comply with **ASTM F1667**.\n- **Minimum attachment** (§1507.2.6):\n  - At least the manufacturer’s requirement, but **not less than 4 fasteners per strip shingle** or **2 per individual shingle**.\n  - For very steep roofs (**> 21:12**), install as required by the manufacturer.\n- **Ice barrier**: Where required, must comply with **§1507.1.2** (§1507.2.7).\n- **Flashings/valleys/drip edge** (§1507.2.8):\n  - Flashing must follow this section **and the manufacturer’s printed instructions**.\n  - **Base/cap flashing**: corrosion-resistant metal (nominal **0.019 in** thick) or qualifying mineral-surfaced roll roofing for base; cap flashing corrosion-resistant metal nominal **0.019 in** (§1507.2.8.1).\n  - **Valleys**: Valley linings installed before shingles; open/closed valley options are permitted, including specified roll-roofing/underlayment options (§1507.2.8.2).\n  - **Drip edge required** at eaves and rake edges (§1507.2.8.3):\n    - Lap adjacent pieces **≥ 2 in**.\n    - Vertical leg **≥ 1.5 in** and extends **≥ 1/4 in below** sheathing; extends back on roof **≥ 2 in**.\n    - Install **underlayment over drip edge at eaves**; install **drip edge over underlayment at rakes**.\n    - Fasten at **≤ 12 in o.c.**\n\n### Underlayment (IBC §1507.1.1)\n- **Underlayment required** for asphalt shingles must comply with §1507.1.1 (referenced by §1507.2.3).\n- **Underlayment types** are specified in **Table 1507.1.1(1)** (by wind speed category).\n- **Underlayment application** for asphalt shingles is in **Table 1507.1.1(2)**:\n  - **2:12 up to 4:12**: **two layers** applied with a **19-in starter strip** at eave; then 36-in sheets overlapping **19 in**; **end laps 4 in** and offset **6 ft**; underlayment must lay flat enough to not prevent shingles sealing.\n  - **≥ 4:12**: **one layer** applied shingle-fashion starting at eave; **lap 2 in**; **end laps 4 in** offset **6 ft**; must not interfere with shingle sealing.\n  - For higher wind category shown, same as low-wind but **all laps ≥ 4 in**.\n- **Underlayment attachment** for asphalt shingles in high-wind category is in **Table 1507.1.1(3)** (grid fastening with cap nails/staples and specified spacings/materials).\n\n### Ice barriers (IBC §1507.1.2)\n- Where there is a **history of ice forming along eaves causing backup of water**, an **ice barrier is required** for asphalt shingles (and other listed coverings) (§1507.1.2).\n- Must be **two layers of underlayment cemented together** *or* a **self-adhering polymer-modified bitumen sheet**.\n- Must extend from the roof’s lowest edges to a point **≥ 24 in inside the exterior wall line** (§1507.1.2).\n- Exception: **Detached accessory structures** without conditioned floor area.\n\nNote: The IBC repeatedly requires following the **manufacturer’s printed installation instructions** in addition to these minimum code provisions (notably for attachment on very steep slopes and for flashings/valleys).'
```

### Results

In [10]:
Markdown(result.output.reasoning)

From the provided context (2021 IBC Chapter 15), composition/asphalt shingles are regulated under IBC §1507.2 with supporting requirements in §1507.1.1 (underlayment types/application/attachment) and §1507.1.2 (ice barriers). Summarize those provisions with key numeric installation criteria.

In [11]:
Markdown(result.output.answer)

From the 2021 International Building Code (IBC), Chapter 15 (Roof Assemblies and Rooftop Structures), ‘composition’ shingles are addressed as **asphalt shingles** (IBC **§1507.2**), with underlayment and ice-barrier rules in **§1507.1.1** and **§1507.1.2**:

### IBC §1507.2 — Asphalt (composition) shingles
- **Deck**: Must be **fastened to solidly sheathed decks** (§1507.2.1).
- **Minimum slope**: Allowed only on roofs **≥ 2:12** (§1507.2.2).
  - For slopes **2:12 up to 4:12**, **double underlayment** is required (§1507.2.2; see underlayment application in §1507.1.1/Table 1507.1.1(2)).
- **Shingle standard**: Shingles must comply with **ASTM D3462** (§1507.2.4).
- **Fasteners (nails)** (§1507.2.5):
  - Must be **galvanized, stainless steel, aluminum, or copper roofing nails**.
  - **Minimum 12-gage** (0.105 in) shank and **minimum 3/8-in head**.
  - Length must penetrate through roofing and **≥ 3/4 in into the roof sheathing** (or **through** the sheathing if sheathing is < 3/4 in).
  - Fasteners must comply with **ASTM F1667**.
- **Minimum attachment** (§1507.2.6):
  - At least the manufacturer’s requirement, but **not less than 4 fasteners per strip shingle** or **2 per individual shingle**.
  - For very steep roofs (**> 21:12**), install as required by the manufacturer.
- **Ice barrier**: Where required, must comply with **§1507.1.2** (§1507.2.7).
- **Flashings/valleys/drip edge** (§1507.2.8):
  - Flashing must follow this section **and the manufacturer’s printed instructions**.
  - **Base/cap flashing**: corrosion-resistant metal (nominal **0.019 in** thick) or qualifying mineral-surfaced roll roofing for base; cap flashing corrosion-resistant metal nominal **0.019 in** (§1507.2.8.1).
  - **Valleys**: Valley linings installed before shingles; open/closed valley options are permitted, including specified roll-roofing/underlayment options (§1507.2.8.2).
  - **Drip edge required** at eaves and rake edges (§1507.2.8.3):
    - Lap adjacent pieces **≥ 2 in**.
    - Vertical leg **≥ 1.5 in** and extends **≥ 1/4 in below** sheathing; extends back on roof **≥ 2 in**.
    - Install **underlayment over drip edge at eaves**; install **drip edge over underlayment at rakes**.
    - Fasten at **≤ 12 in o.c.**

### Underlayment (IBC §1507.1.1)
- **Underlayment required** for asphalt shingles must comply with §1507.1.1 (referenced by §1507.2.3).
- **Underlayment types** are specified in **Table 1507.1.1(1)** (by wind speed category).
- **Underlayment application** for asphalt shingles is in **Table 1507.1.1(2)**:
  - **2:12 up to 4:12**: **two layers** applied with a **19-in starter strip** at eave; then 36-in sheets overlapping **19 in**; **end laps 4 in** and offset **6 ft**; underlayment must lay flat enough to not prevent shingles sealing.
  - **≥ 4:12**: **one layer** applied shingle-fashion starting at eave; **lap 2 in**; **end laps 4 in** offset **6 ft**; must not interfere with shingle sealing.
  - For higher wind category shown, same as low-wind but **all laps ≥ 4 in**.
- **Underlayment attachment** for asphalt shingles in high-wind category is in **Table 1507.1.1(3)** (grid fastening with cap nails/staples and specified spacings/materials).

### Ice barriers (IBC §1507.1.2)
- Where there is a **history of ice forming along eaves causing backup of water**, an **ice barrier is required** for asphalt shingles (and other listed coverings) (§1507.1.2).
- Must be **two layers of underlayment cemented together** *or* a **self-adhering polymer-modified bitumen sheet**.
- Must extend from the roof’s lowest edges to a point **≥ 24 in inside the exterior wall line** (§1507.1.2).
- Exception: **Detached accessory structures** without conditioned floor area.

Note: The IBC repeatedly requires following the **manufacturer’s printed installation instructions** in addition to these minimum code provisions (notably for attachment on very steep slopes and for flashings/valleys).

In [12]:
result.usage.__dict__

{'input_tokens': 106404,
 'cache_write_tokens': 0,
 'cache_read_tokens': 82944,
 'output_tokens': 3582,
 'input_audio_tokens': 0,
 'cache_audio_read_tokens': 0,
 'output_audio_tokens': 0,
 'details': {'accepted_prediction_tokens': 0,
  'audio_tokens': 0,
  'reasoning_tokens': 0,
  'rejected_prediction_tokens': 0},
 'requests': 10,
 'tool_calls': 0}

In [ ]:
result = await rlm.run(
    query="Please extract all guidelines around water damage prevention steps for residential construction.",
    context=pages,
)

In [19]:
display(Markdown(result.render_trajectory()))

### Iteration 1

> **Reasoning:** Need to explore the provided `query` and `context` structures first (types, lengths, sample) before attempting an answer.

```python
import json, re
print('query type:', type(query))
print('query:', query)
print('context type:', type(context))
try:
    print('context len:', len(context))
except Exception as e:
    print('context len error:', e)

# Show a small preview
if isinstance(context, str):
    print('context chars:', len(context))
    print('context preview:', context[:500])
elif isinstance(context, list):
    print('first 3 items types/preview:')
    for i, item in enumerate(context[:3]):
        print('--- item', i, 'type', type(item))
        s = item if isinstance(item, str) else json.dumps(item)[:500]
        print(s[:500])
else:
    print('context preview:', str(context)[:500])
```

**Output:**
```
query type: <class 'str'>
query: Please extract all guidelines around water damage prevention steps for residential construction.
context type: <class 'list'>
context len: 833
first 3 items types/preview:
--- item 0 type <class 'str'>
101167924

--- item 1 type <class 'str'>
IBC
®
INTERNATIONAL  
BUILDING  
CODE
® 
A Member of the International Code Family®
Copyright © 2020 ICC.  ALL RIGHTS RESERVED. Accessed by Nathan Vander Plaats (nvp@cityoflennoxsd.com), (-) Order Number #101167924 on Oct 14, 2021 06:19 AM (PDT) pursuant to License Agreement
with ICC.  No further reproduction, no further reproductions by any third party, or distribution authorized.  Single user only, copying and networking prohibited. ANY UNAUTHORIZED REPRODUCTION OR
DISTRIBUTION IS A VIOLATION 
--- item 2 type <class 'str'>
2021 International Building Code
Date of First Publication: October 23, 2020
First Printing: October 2020
ISBN: 978-1-60983-955-0 (soft-cover edition)
ISBN: 978-1-60983-956-7 (loose-leaf edition)
COPYRIGHT  2020
by
INTERNATIONAL CODE COUNCIL, INC.
ALL RIGHTS RESERVED. This 2021 International Building Code® is a copyrighted work owned by the International Code
Council, Inc. (“ICC”). Without advance written permission from the ICC, no part of this book may be reproduced, distributed or
transmitt

```

---

### Iteration 2

> **Reasoning:** We need extract guidelines around water damage prevention steps for residential construction from a long list context (833 strings). We'll search within context for water/moisture/damp/water-resistive/vapor/flashing/drainage etc. Use Python to locate relevant indices/snippets then use llm_query over selected excerpts to synthesize guidelines. First, find hits with keyword scanning.

```python
import re

def find_hits(patterns, max_hits=80):
    hits=[]
    for i,txt in enumerate(context):
        if not isinstance(txt,str):
            continue
        low=txt.lower()
        if any(p in low for p in patterns):
            hits.append(i)
            if len(hits)>=max_hits:
                break
    return hits

patterns=[
    'water', 'moisture', 'damp', 'leak', 'drain', 'flashing', 'roof', 'vapor', 'vapour',
    'water-resistive', 'weather-resistive', 'dampproof', 'waterproof', 'condensation',
    'crawl space', 'basement', 'foundation', 'sill', 'gutter', 'downspout', 'site drainage',
    'plumbing', 'shower', 'tub', 'roof drain', 'overflow'
]

hits=find_hits(patterns, max_hits=120)
print('hit count (capped):', len(hits))
print('first 30 hit indices:', hits[:30])

# print small snippets around first 8 hits for orientation
for idx in hits[:8]:
    txt=context[idx]
    snippet=re.sub(r'\s+',' ',txt)[:400]
    print('\n--- idx',idx,'---')
    print(snippet)
```

**Output:**
```
hit count (capped): 120
first 30 hit indices: [3, 5, 6, 12, 14, 15, 16, 17, 18, 19, 21, 23, 24, 25, 26, 27, 28, 29, 31, 34, 35, 36, 37, 38, 39, 40, 43, 44, 45, 46]

--- idx 3 ---
2021 INTERNATIONAL BUILDING CODE® iii PREFACE Introduction The International Building Code® (IBC®) establishes minimum requirements for building systems using prescriptive and performance-related provisions. It is founded on broad-based principles that make possible the use of new materials and new building designs. This 2021 edition is fully compatible with all of the International Codes® (I-Code

--- idx 5 ---
2021 INTERNATIONAL BUILDING CODE® v Code Development Committee Responsibilities (Letter Designations in Front of Section Numbers) As mentioned in the preceding material, in each code development cycle, code change proposals to this code are considered at the Committee Action Hearings by 11 different code development commit- tees. Code change proposals to sections of the code that are preceded by a

--- idx 6 ---
2021 INTERNATIONAL BUILDING CODE® vi CODE DEVELOPMENT HEARINGS Note: Proposed changes to the ICCPC will be heard by the code development committee noted in brackets [ ] in the text of the ICCPC. Marginal Markings Solid vertical lines in the margins within the body of the code indicate a technical change from the requirements of the 2018 edition. Deletion indicators in the form of an arrow ( ) are 

--- idx 12 ---
2021 INTERNATIONAL BUILDING CODE® xii IMC Correlated Topics The IBC requirements for smoke control systems, and smoke and fire dampers are directly correlated to the requirements of the IMC. IBC Chapter 28 is a reference to the IMC and the IFGC for chimneys, fireplaces and barbecues, and all aspects of mechanical systems. The following table shows chapters/ sections of the IBC that are correlated 

--- idx 14 ---
2021 INTERNATIONAL BUILDING CODE® xiv Chapter 5 General Building Heights and Areas Chapter 5 contains the provisions that regulate the minimum type of construction for area limits and height limits based on the occupancy of the building. Height and area increases (including allowances for basements, mezzanines and equipment platforms) are permitted based on open frontage for fire department access

--- idx 15 ---
2021 INTERNATIONAL BUILDING CODE® xv Chapter 9 Fire Protection and Life Safety Systems Chapter 9 prescribes the minimum requirements for active systems of fire protection equipment to perform the following functions: detect a fire; alert the occupants or fire department of a fire emer- gency; and control smoke and control or extinguish the fire. Generally, the requirements are based on the occupan

--- idx 16 ---
2021 INTERNATIONAL BUILDING CODE® xvi Chapter 13 Energy Efficiency The purpose of Chapter 13 is to provide minimum design requirements that will promote efficient uti- lization of energy in buildings. The requirements are directed toward the design of building envelopes with adequate thermal resistance and low air leakage, and toward the design and selection of mechanical, water heating, electrica

--- idx 17 ---
2021 INTERNATIONAL BUILDING CODE® xvii Chapter 18 Soils and Foundations Chapter 18 provides criteria for geotechnical and structural considerations in the selection, design and installation of foundation systems to support the loads from the structure above. This chapter includes requirements for soils investigation and site preparation for receiving a foundation, including the allowed load-bearin

```

---

### Iteration 3

> **Reasoning:** Need to narrow to residential/water-damage related provisions; first locate the most relevant context segments by stronger keyword/regex and extract surrounding text for synthesis.

```python
import re, json
# stronger patterns to focus on water damage prevention in building envelope/foundations/plumbing
kw = [
    r"water[- ]resistive", r"weather[- ]resistive", r"flashing", r"weep", r"drainage", r"dampproof", r"damp[- ]proof",
    r"waterproof", r"vapor retarder", r"vapou?r retarder", r"vapor barrier", r"condensation",
    r"roof drain", r"gutters?", r"downspouts?", r"overflow", r"slope", r"foundation drainage",
    r"crawl\s*space", r"basement", r"site drainage", r"ground surface", r"capillary", r"sill plate",
    r"shower", r"bathtub", r"tub", r"plumbing leak", r"backflow", r"trap", r"pan"
]
rx = re.compile("|".join(kw), re.IGNORECASE)

hit_idxs=[]
for i, txt in enumerate(context):
    if isinstance(txt,str) and rx.search(txt):
        hit_idxs.append(i)

print('total focused hits:', len(hit_idxs))
print('first 40 hit idxs:', hit_idxs[:40])

# show up to 15 best-looking snippets (longer text tends to be actual code sections)
snips=[]
for i in hit_idxs:
    t=context[i]
    if not isinstance(t,str):
        continue
    s=re.sub(r"\s+"," ",t).strip()
    snips.append((len(s), i, s[:500]))

snips.sort(reverse=True)
for L,i,s in snips[:15]:
    print(f"\n--- idx {i} (len {L}) ---\n{s}")

SAVE(hit_idxs=hit_idxs, top_snips=snips[:50])
```

**Output:**
```
total focused hits: 622
first 40 hit idxs: [9, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21, 23, 24, 25, 26, 27, 28, 29, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53]

--- idx 784 (len 7420) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® Guards. . . . . . . . . . . . . . . . . . 1015.2.1, 2406.4.4, 2407 Handrail. . . . . . . . . . . . . . . . . . . . . . . . . 1011.11, 2407 Identification . . . . . . . . . . . . . . . . . . . . . 2403.1, 2406.3 Impact loads . . . . . . . . . . . . . 2406.1, 2408.2.1, 2408.3 Impact resistant. . . . . . . . . . . . . . . . . . . . . . . . . 1609.2 Jalousies . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2403.5 Label/identification. . . . . . . .

--- idx 790 (len 7243) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® LIVE/WORK UNITS . . . . . . . . . . . . . . . . 202, 310.3, 419 Accessibility. . . . . . . . . . . . . . . . . . . . . . . . . 1108.6.2.1 Separation. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 508.1 LOAD AND RESISTANCE FACTOR DESIGN (LRFD) . . . . . . . . . . . . . . . . 1602.1 Factored load . . . . . . . . . . . . . . . . . . . . . . 202, 1604.2 Limit state . . . . . . . . . . . . . . . . . . . . . . . . . 202, 1604.2 Load combinations . . 

--- idx 799 (len 7103) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® Standpipe control valve . . . . . . . . . . . . . . . . . . 905.7.1 Toilet room . . . . . . . .1112.1, 1112.3, 2902.4, 2902.4.1 Transportation. . . . . . . . . . . . . . . . . . E108.4, E109.2.2 Variable message . . . . . . . . . . . . . . . . . . . . . . . .1112.5 Walls . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 703.4, H111 SITE DRAWINGS . . . . . . . . . . . . . . . . . . . . . . . . 107.2.6 SITE WORK. . . . . . . . . . . . . . . . . . . .

--- idx 775 (len 7016) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® Surface bonding . . . . . . . . . . . . . . . . .2103.2.2, 2109.2 Wood support . . . . . . . . . . . . . . . . . . . . . . . . . .2304.13 CONCRETE ROOF TILE. . . . . . . . . . . . . . . . . . . .1507.3 Wind resistance . . . . . . . . . . . . . . . . .1504.3, 1609.5.3 CONDOMINIUM (see APARTMENT HOUSES) CONDUIT, PENETRATION PROTECTION . . . . . . . 713.3, 1023.5 CONFLICTS IN CODE . . . . . . . . . . . . . . . 102, 104.10.1 CONGREGATE LIVING FACILITIES . . . 

--- idx 798 (len 6967) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® Grandstands (see GRANDSTANDS) Guards. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1030.17 Live load . . . . . . . . . . . . . . . . . . . . . . . . . Table 1607.1 Occupant load . . . . . . . . . . . . . . . . . . . . . . . . . . 1004.6 Stability . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1030.15 Temporary. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 108 SECURITY GLAZING . . . . . . . . . . . . . . . . . . 

--- idx 794 (len 6962) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® PARTICLEBOARD . . . . . . . . . . . . . . . . . . . . . . . . . . 202 Draftstopping. . . . . . . . . . . . . . . . . . . . . . . . . . . 718.3.1 Moisture protection . . . . . . . . . . . . . . . . 1402.2, 1404.2 Quality . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2303.1.8 Veneer. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1404.5 Wall bracing. . . . . . . . . . . . . . . . . . . . . . . . . . . 2308.6.3 PARTITIONS Fabric

--- idx 797 (len 6947) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® ROOF CONSTRUCTION Construction walkways . . . . . . . . . . . . . . . . . . . .3306.7 Coverings (see ROOF COVERINGS) . . . . . . 1609.5.2 Deck . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1609.5.1 Draftstopping . . . . . . . . . . . . . . . . . . . . . . . . . . . .718.4 Fire resistance . . . . . . . . . . . . . . . . . . . . . . . Table 601 Fireblocking . . . . . . . . . . . . . . . . . . . . . . . . . . . . .718.2 Live loads . . . . . 

--- idx 788 (len 6947) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® Occupancy conditions . . . . . . . . . . . . . 308.2.1, 308.2.2 Occupancy exceptions . . . . . . . . . . . . 308.2.3, 308.2.4 Separation. . . . . . . . . . . . . . . . . . . 420.2, 420.3, 420.6 Spaces open to corridors . . . . . . . . . . . . . 420.7, 420.8 Sprinkler protection . . . . . . . . . 420.5, 903.2.6, 903.3.2 INSTITUTIONAL I-2 [see INSTITUTIONAL OCCU- PANCY (see GROUP I)]. . . . . . . . . . . . . . . 308.3, 407 Accessibility. . . . . 1106.4, 1106

--- idx 796 (len 6929) ---
INDEX 2021 INTERNATIONAL BUILDING CODE® REINFORCEMENT Masonry. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2103.4 Shotcrete. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 1908 RELIGIOUS WORSHIP, PLACES OF Accessibility. . . . . . . . . . . . . . . . . . . 1103.2.8, 1109.2.4 Alarms and detection. . . . . . . . . . . . . . . . . . . . . 907.2.1 Balcony . . . . . . . . . . . . . . . . . . . . . . . 1030.5, 1109.2.4 Classification . . . . . . . . . . . . . . 303.1.4, 

--- idx 791 (len 6904) ---
INDEX 20
... [truncated]
```

---

### Iteration 4

> **Reasoning:** We need extract water damage prevention guidelines from long IBC context list. Next step: locate code sections likely containing moisture protection requirements (Chapter 14 exterior walls, foundations drainage, roofs, plumbing wet areas). We'll use keyword-based index finding for specific section numbers (e.g., 1402.2 moisture protection, 1503 weather protection, 1805 foundation waterproofing/dampproofing, 2304.12 protection against decay, 2509 gypsum wet areas). Then pull surrounding items and use llm_query later to synthesize.

```python
import re

# Find indices containing likely relevant section numbers/headings
sec_patterns = [
    r"\b1402\.2\b", r"\b1403\b", r"\b1404\b", r"\b1405\b",  # exterior walls / weather protection
    r"\b1503\b", r"\b1504\b", r"\b1507\b", r"\b1509\b",          # roof / weather protection
    r"\b1805\b", r"\b1804\.4\b", r"\b1807\b", r"\b1808\b",     # foundation walls / dampproofing / drainage
    r"\b2304\.12\b", r"\b2304\.11\b", r"\b2304\.13\b",          # wood protection/decay
    r"\b2509\b", r"\b2510\b",                                       # gypsum in wet areas
    r"\b2902\b", r"\b2903\b"                                        # plumbing fixtures (sometimes wet area notes)
]
rx = re.compile("|".join(sec_patterns))

hits=[]
for i,t in enumerate(context):
    if isinstance(t,str) and rx.search(t):
        hits.append(i)

print('section-number hits:', len(hits))
print('first 80 idx:', hits[:80])

# print small previews around first 12 hits
for idx in hits[:12]:
    s=re.sub(r"\s+"," ",context[idx]).strip()
    print(f"\n--- idx {idx} ---\n{s[:700]}")

SAVE(section_hits=hits)
```

**Output:**
```
section-number hits: 158
first 80 idx: [8, 17, 25, 26, 27, 55, 100, 155, 156, 158, 164, 165, 197, 198, 199, 200, 211, 212, 256, 258, 275, 353, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392, 393, 395, 396, 397, 398, 399, 400, 401, 402, 403, 404, 405, 406, 407, 408, 409, 410, 411, 413, 420, 446, 447, 475, 487, 488, 489, 490, 491, 492, 493, 494, 495, 496, 497, 498, 499, 500, 501, 502, 524, 525, 542, 560, 561, 562, 563, 584, 587]

--- idx 8 ---
2021 INTERNATIONAL BUILDING CODE® viii Coordination of the International Codes The coordination of technical provisions is one of the strengths of the ICC family of model codes. The codes can be used as a complete set of complementary documents, which will provide users with full integration and coordination of technical provisions. Individual codes can also be used in subsets or as stand-alone documents. To make sure that each individual code is as complete as possible, some tech- nical provisions that are relevant to more than one subject area are duplicated in some of the model codes. This allows users maximum flexibility in their application of the I-Codes. Italicized Terms Terms italici

--- idx 17 ---
2021 INTERNATIONAL BUILDING CODE® xvii Chapter 18 Soils and Foundations Chapter 18 provides criteria for geotechnical and structural considerations in the selection, design and installation of foundation systems to support the loads from the structure above. This chapter includes requirements for soils investigation and site preparation for receiving a foundation, including the allowed load-bearing values for soils and for protecting the foundation from water intrusion. Section 1808 addresses the basic requirements for all foundation types. Later sections address foundation requirements that are specific to shallow foundations and deep foundations. Due care must be exer- cised in the plannin

--- idx 25 ---
TABLE OF CONTENTS 2021 INTERNATIONAL BUILDING CODE® 1005 Means of Egress Sizing. . . . . . . . . . . . . . . . . . . . 10-4 1006 Number of Exits and Exit Access Doorways . . . . . . . . . . . . . . . . . . 10-5 1007 Exit and Exit Access Doorway Configuration . . . . . . . . . . . . . . . . . 10-8 1008 Means of Egress Illumination. . . . . . . . . . . . . . . 10-9 1009 Accessible Means of Egress . . . . . . . . . . . . . . . 10-10 1010 Doors, Gates and Turnstiles . . . . . . . . . . . . . . . 10-12 1011 Stairways. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 10-22 1012 Ramps . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 10-26 1013 Exit Signs . . . . . . . . . . 

--- idx 26 ---
TABLE OF CONTENTS 2021 INTERNATIONAL BUILDING CODE® 1511 Rooftop Structures . . . . . . . . . . . . . . . . . . . . . . 15-17 1512 Reroofing . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 15-19 CHAPTER 16 STRUCTURAL DESIGN . . . . . . . . 16-1 Section 1601 General . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 16-1 1602 Notations. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 16-1 1603 Construction Documents . . . . . . . . . . . . . . . . . . 16-1 1604 General Design Requirements . . . . . . . . . . . . . . 16-2 1605 Load Combinations. . . . . . . . . . . . . . . . . . . . . . . 16-6 1606 Dead Loads. . . . . . . . . . . . . . . . . . . . . . . . . .

--- idx 27 ---
TABLE OF CONTENTS 2021 INTERNATIONAL BUILDING CODE® 2204 Connections . . . . . . . . . . . . . . . . . . . . . . . . . . . . 22-1 2205 Structural Steel . . . . . . . . . . . . . . . . . . . . . . . . . . 22-1 2206 Composite Structural Steel and Concrete Structures. . . . . . . . . . . . . . . . . . . . . 22-2 2207 Steel Joists. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 22-2 2208 Steel Cable Structures. . . . . . . . . . . . . . . . . . . . . 22-3 2209 Steel Storage Racks . . . . . . . . . . . . . . . . . . . . . . 22-3 2210 Cold-formed Steel. . . . . . . . . . . . . . . . . . . . . . . . 22-3 2211 Cold-formed Steel Light-frame Construction. . . 22-4 CHAPTER 23 WOOD . . . . . 

--- idx 55 ---
DEFINITIONS 2021 INTERNATIONAL BUILDING CODE® [F] FIRE PROTECTION SYSTEM. Approved devices, equipment and systems or combinations of systems used to detect a fire, activate an alarm, extinguish or control a fire, control or manage smoke and products of a fire or any combi- nation thereof. [BF] FIRE PROTECTIVE CURTAIN ASSEMBLY. An assembly consisting of a fabric curtain, a bottom bar, guides, a coil, and an operating and closing system. [BF] FIRE RESISTANCE. That property of materials or their assemblies that prevents or retards the passage of exces- sive heat, hot gases or flames under conditions of use. [F] FIRE SAFETY FUNCTIONS. Building and fire control functions that are intended to incr

--- idx 100 ---
SPECIAL DETAILED REQUIREMENTS BASED ON OCCUPANCY AND USE 2021 INTERNATIONAL BUILDING CODE® 506 as modified by Section 507. Roof parking is permitted. 406.6.2 Ventilation. A mechanical ventilation system and exhaust system shall be provided in accord
... [truncated]
```

---

### Iteration 5

> **Reasoning:** We need to extract guidelines on water damage prevention for residential construction from a long IBC context list. Next step is to identify and pull the most relevant code sections (exterior walls/weather protection, roofs/flashings, foundations/dampproofing/drainage, wood decay protection, wet areas) by scanning for key section headings and then summarizing them via sub-LLM. We'll first locate candidate indices for these topics and extract surrounding text windows for later synthesis.

```python
import re

# Target topics and section identifiers common to IBC moisture/water protection
patterns = {
    'Exterior walls / weather protection (Ch 14)': [r"\bCHAPTER\s+14\b", r"\b1402\.2\b", r"MOISTURE\s+PROTECTION", r"WEATHER\s+PROTECTION", r"WATER[- ]RESISTIVE"],
    'Roofing / flashing (Ch 15)': [r"\bCHAPTER\s+15\b", r"\b1503\b", r"FLASHING", r"ROOF\s+DRAIN", r"COPING"],
    'Foundations / damp-proofing / waterproofing (Ch 18)': [r"\b1805\b", r"DAMPPROOF", r"WATERPROOF", r"FOUNDATION\s+DRAIN", r"DRAINAGE\s+SYSTEM"],
    'Wood decay / moisture in wood (Ch 23)': [r"\b2304\.12\b", r"\b2304\.11\b", r"DECAY", r"PRESERVATIVE\s*TREATED", r"WOOD\s+IN\s+CONTACT\s+WITH\s+THE\s+GROUND"],
    'Wet areas / interior finishes': [r"\b2509\b", r"CEMENT\s*PLASTER", r"WATER\s*RESISTANT\s*GYPSUM", r"SHOWER", r"BATHTUB"],
}

hits_by_topic = {k: [] for k in patterns}
for i, t in enumerate(context):
    if not isinstance(t, str):
        continue
    for topic, pats in patterns.items():
        rx = re.compile("|".join(pats), re.IGNORECASE)
        if rx.search(t):
            hits_by_topic[topic].append(i)

for topic, idxs in hits_by_topic.items():
    print(topic, 'hit count:', len(idxs), 'first idxs:', idxs[:10])

# Build excerpt windows around first few hits per topic for later summarization
windows = []
for topic, idxs in hits_by_topic.items():
    for idx in idxs[:5]:
        start = max(0, idx-1)
        end = min(len(context), idx+2)
        excerpt = "\n".join(context[start:end])
        windows.append((topic, idx, excerpt))

print('\nTotal windows:', len(windows))
for topic, idx, excerpt in windows[:6]:
    ex = re.sub(r"\s+", " ", excerpt)[:500]
    print(f"\n--- {topic} @ idx {idx} ---\n{ex}")

SAVE(hits_by_topic=hits_by_topic, windows=windows)
```

**Output:**
```
Exterior walls / weather protection (Ch 14) hit count: 60 first idxs: [16, 18, 19, 25, 36, 45, 46, 67, 68, 74]
Roofing / flashing (Ch 15) hit count: 56 first idxs: [15, 16, 21, 25, 36, 65, 67, 68, 138, 156]
Foundations / damp-proofing / waterproofing (Ch 18) hit count: 39 first idxs: [26, 39, 71, 73, 74, 119, 120, 216, 353, 378]
Wood decay / moisture in wood (Ch 23) hit count: 37 first idxs: [8, 63, 100, 155, 156, 158, 165, 197, 198, 256]
Wet areas / interior finishes hit count: 48 first idxs: [27, 47, 74, 182, 199, 201, 202, 208, 209, 210]

Total windows: 25

--- Exterior walls / weather protection (Ch 14) @ idx 16 ---
2021 INTERNATIONAL BUILDING CODE® xv Chapter 9 Fire Protection and Life Safety Systems Chapter 9 prescribes the minimum requirements for active systems of fire protection equipment to perform the following functions: detect a fire; alert the occupants or fire department of a fire emer- gency; and control smoke and control or extinguish the fire. Generally, the requirements are based on the occupancy, the height and the area of the building, because these are the factors that most affect fire-fig

--- Exterior walls / weather protection (Ch 14) @ idx 18 ---
2021 INTERNATIONAL BUILDING CODE® xvii Chapter 18 Soils and Foundations Chapter 18 provides criteria for geotechnical and structural considerations in the selection, design and installation of foundation systems to support the loads from the structure above. This chapter includes requirements for soils investigation and site preparation for receiving a foundation, including the allowed load-bearing values for soils and for protecting the foundation from water intrusion. Section 1808 addresses th

--- Exterior walls / weather protection (Ch 14) @ idx 19 ---
2021 INTERNATIONAL BUILDING CODE® xviii Chapter 23 Wood This chapter provides minimum requirements for the design of buildings and structures that use wood and wood-based products. The chapter is organized around three design methodologies: allowable stress design (ASD), load and resistance factor design (LRFD) and conventional light-frame construc- tion. Included in this chapter are references to design and manufacturing standards for various wood and wood-based products; general construction r

--- Exterior walls / weather protection (Ch 14) @ idx 25 ---
TABLE OF CONTENTS 2021 INTERNATIONAL BUILDING CODE® CHAPTER 5 GENERAL BUILDING HEIGHTS AND AREAS . . . . . . . . . . . 5-1 Section 501 General. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5-1 502 Building Address . . . . . . . . . . . . . . . . . . . . . . . . . . 5-1 503 General Building Height and Area Limitations. . . 5-1 504 Building Height and Number of Stories . . . . . . . . 5-2 505 Mezzanines and Equipment Platforms . . . . . . . . . 5-2 506 Building Area . . . . . 

--- Exterior walls / weather protection (Ch 14) @ idx 36 ---
SCOPE AND ADMINISTRATION 2021 INTERNATIONAL BUILDING CODE® Electrical: 1. Repairs and maintenance: Minor repair work, including the replacement of lamps or the connec- tion of approved portable electrical equipment to approved permanently installed receptacles. 2. Radio and television transmitting stations: The provisions of this code shall not apply to electrical equipment used for radio and television transmis- sions, but do apply to equipment and wiring for a power supply and the installation

--- Roofing / flashing (Ch 15) @ idx 15 ---
2021 INTERNATIONAL BUILDING CODE® xiv Chapter 5 General Building Heights and Areas Chapter 5 contains the provisions that regulate the minimum type of construction for area limits and height limits based on the occupancy of the building. Height and area increases (including allowances for basements, mezzanines and equipment platforms) are permitted based on open frontage for fire department access, separation and the type of sprinkler protection provided (Sections 503-506, 510). These thresholds

```

---

### Iteration 6

> **Reasoning:** We need to extract actual water-damage prevention guidelines from the IBC text. The current windows were poor (mostly ToC/preface). Next step: find the actual detailed sections by searching for headings like 'Section 1404', '1404.2', '1503.2 Flashing', '1805.4 Foundation drainage', etc., then build larger excerpts around those indices and use llm_query to summarize. We'll first locate best indices by regex for '1404.2', '1503.2', '1805.4', '2304.12', '2509.2' etc and preview them.

```python
import re

# Find indices that look like actual code sections/subsections for moisture protection
sec_rx = re.compile(r"\b(1402\.2|1403\.|1404\.|1405\.|1503\.|1504\.|1507\.|1804\.|1805\.|2304\.(11|12|13)|2509\.|2510\.|2704\.|2902\.)\d*\b")

candidates=[]
for i,t in enumerate(context):
    if isinstance(t,str) and sec_rx.search(t):
        # score by presence of key moisture terms
        low=t.lower()
        score = sum(k in low for k in [
            'moisture','water','weather','damp','dampproof','waterproof','flashing','weep','drain','vapor','condensation','shower','bathtub'
        ])
        candidates.append((score, len(t), i, t))

candidates.sort(reverse=True)
print('candidate count:', len(candidates))
print('top 20 candidates (score,len,idx):')
for score,L,i,_ in candidates[:20]:
    print(score, L, i)

# preview top 8 with some whitespace normalized
for score,L,i,t in candidates[:8]:
    s=re.sub(r"\s+"," ",t).strip()
    print(f"\n--- idx {i} score {score} len {L} ---\n{s[:900]}")

SAVE(candidates=candidates)
```

**Output:**
```
[Error] TypeError: unsupported operand type(s) for +: 'int' and 'bool'
```

---

### Iteration 7

> **Reasoning:** Fix the TypeError (bool summation) and continue by identifying the highest-relevance IBC section excerpts for moisture/water protection so we can synthesize residential water-damage prevention guidelines.

```python
import re

sec_rx = re.compile(r"\b(1402\.2|1403\.|1404\.|1405\.|1503\.|1504\.|1507\.|1804\.|1805\.|2304\.(11|12|13)|2509\.|2510\.|2704\.|2902\.)\d*\b")

candidates=[]
for i,t in enumerate(context):
    if isinstance(t,str) and sec_rx.search(t):
        low=t.lower()
        score = sum(int(k in low) for k in [
            'moisture','water','weather','damp','dampproof','waterproof','flashing','weep','drain','vapor','condensation','shower','bathtub'
        ])
        candidates.append((score, len(t), i, t))

candidates.sort(reverse=True)
print('candidate count:', len(candidates))
print('top 20 candidates (score,len,idx):')
for score,L,i,_ in candidates[:20]:
    print(score, L, i)

for score,L,i,t in candidates[:10]:
    s=re.sub(r"\s+"," ",t).strip()
    print(f"\n--- idx {i} score {score} len {L} ---\n{s[:900]}")

SAVE(candidates=candidates)
```

**Output:**
```
candidate count: 124
top 20 candidates (score,len,idx):
7 5604 631
7 4497 408
6 6148 490
6 6139 632
5 7269 790
5 6569 776
5 6435 390
5 6219 491
5 6084 386
5 5477 395
5 5431 630
5 4924 383
4 6632 806
4 6468 388
4 6374 783
4 6364 387
4 6362 389
4 5905 384
4 5744 563
4 5351 489

--- idx 631 score 7 len 5604 ---
GYPSUM BOARD, GYPSUM PANEL PRODUCTS AND PLASTER 2021 INTERNATIONAL BUILDING CODE® Type I. The individual layers shall be installed independently such that each layer provides a separate continuous plane and any flashing, installed in accordance with Section 1404.4 and intended to drain to the water-resistive barrier, is directed between the layers. 2. The water-resistive barrier shall be 60-minute Grade D paper or have a water resistance equal to or greater than one layer of water-resistive barrier complying with ASTM E2556, Type II. The water-resistive barrier shall be separated from the stucco by a layer of foam plastic insulating sheath- ing or other nonwater absorbing layer, or a drainage space. 2510.6.2 Moist or marine climates. In moist (A) or marine (C) climate zones, water-resistive barrier shall comply with one of the following: 1. In addition to complying with Item 1 or 2 of Se

--- idx 408 score 7 len 4497 ---
ROOF ASSEMBLIES AND ROOFTOP STRUCTURES 2021 INTERNATIONAL BUILDING CODE® not less than 11 inches (279 mm) from the centerline each way and have a splash diverter rib not less than 1 inch (25 mm) high at the flow line formed as part of the flashing. Sections of flashing shall have an end lap of not less than 4 inches (102 mm). For roof slopes of three units vertical in 12 units horizontal (25-percent slope) and over, the valley flashing shall have a 36-inch-wide (914 mm) underlayment of either one layer of Type I underlayment running the full length of the valley or a self-adhering polymer-modified bitumen sheet bearing a label indicat- ing compliance with ASTM D1970, in addition to other required underlayment. In areas where the average daily temperature in January is 25°F (-4°C) or less or where there is a possibility of ice forming along the eaves caus- ing a backup of water, the metal

--- idx 490 score 6 len 6148 ---
SOILS AND FOUNDATIONS 2021 INTERNATIONAL BUILDING CODE® dation shall be permitted to be reduced to not less than 1 unit vertical in 48 units horizontal (2- percent slope). 2. Impervious surfaces shall be permitted to be sloped less than 2 percent where the surface is a door landing or ramp that is required to comply with Section 1010.1.4, 1012.3 or 1012.6.1. The procedure used to establish the final ground level adjacent to the foundation shall account for additional settle- ment of the backfill. 1804.5 Grading and fill in flood hazard areas. In flood hazard areas established in Section 1612.3, grading, fill, or both, shall not be approved: 1. Unless such fill is placed, compacted and sloped to minimize shifting, slumping and erosion during the rise and fall of flood water and, as applicable, wave action. 2. In floodways, unless it has been demonstrated through hydrologic and hydraulic a

--- idx 632 score 6 len 6139 ---
GYPSUM BOARD, GYPSUM PANEL PRODUCTS AND PLASTER 2021 INTERNATIONAL BUILDING CODE® (51 mm) above paved areas and be of a type that will allow trapped water to drain to the exterior of the build- ing. The water-resistive barrier shall lap the attachment flange. The exterior lath shall cover and terminate on the attachment flange of the weep screed. 2512.2 Plasticity agents. Only approved plasticity agents and approved amounts thereof shall be added to Portland cement or blended cements. Where plastic cement or masonry cement is used, additional lime or plasticizers shall not be added. Hydrated lime or the equivalent amount of lime putty used as a plasticizer is permitted to be added to cement plaster or cement and lime plaster in an amount not to exceed that set forth in ASTM C926. 2512.3 Limitations. Gypsum plaster shall not be used on exterior surfaces. 2512.4 Cement plaster. Plaster coa

--- idx 790 score 5 len 7269 ---
INDEX 2021 INTERNATIONAL BUILDING CODE® LIVE/WORK UNITS . . . . . . . . . . . . . . . . 202, 310.3, 419 Accessibility. . . . . . . . . . . . . . . . . . . . . . . . . 1108.6.2.1 Separation. . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 508.1 LOAD AND RESISTANCE FACTOR DESIGN (LRFD) . . . . . . . . . . . . . . . . 1602.1 Factored load . . . . . . . . . . . . . . . . . . . . . . 202, 1604.2 Limit state . . . . . . . . . . . . . . . . . . . . . . . . . 202, 1604.2 Load combinations . . . . . . . . . . . . . . . . . . . . . . . . 1605 Load factor . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 202 Resistance factor. . . . . . . . . . . . . . . . . . . . . . . . . . . 202 Wood design . . . . . . . . . . . . . . . . . . . . . . 2302.1, 2307 LOAD COMBINATIONS . . . . . . . . . . . . . . . . . . . . . 1605 Allowable stress design. . . . . . . . . . . . . . 1605, 1605.2 

--- idx 776 score 5 len 6569 ---
INDEX 2021 INTE
... [truncated]
```

---

### Iteration 8

> **Reasoning:** Continue iterative extraction: pull full text excerpts for the most relevant section indices (exterior walls vapor retarders, flashing/windows, foundation dampproofing/drainage, roof flashing/ice barrier, stucco WRB/drainage) and then use sub-LLM to synthesize water-damage prevention guidelines geared to residential construction.

```python
import re

# choose key indices from prior candidate list (high scores, non-index)
key_idxs = [386, 390, 408, 491, 631, 632, 490, 395]

# build larger excerpts by concatenating neighboring items until ~12k chars per excerpt
excerpts=[]
for idx in key_idxs:
    parts=[]
    total=0
    # expand outward window
    for j in range(max(0, idx-2), min(len(context), idx+3)):
        if isinstance(context[j], str):
            parts.append(context[j])
            total += len(context[j])
    excerpt='\n'.join(parts)
    excerpts.append((idx, excerpt))
    print('\n==== excerpt idx', idx, 'chars', len(excerpt), '====')
    print(re.sub(r'\s+',' ', excerpt)[:700])

# Now ask sub-LLM to extract actionable water-damage prevention guidelines from each excerpt
prompts=[]
for idx, ex in excerpts:
    prompts.append(f"You are extracting code-based guidelines for preventing water damage in residential construction. \
From the excerpt below, list ONLY actionable requirements/steps related to moisture/water protection (e.g., grading/drainage, dampproofing/waterproofing, vapor retarders, flashing, water-resistive barriers, roof underlayment/ice barriers, weep screeds). \
If any prescriptive measurements (e.g., slopes, lap lengths, thickness) appear, include them. \
Cite section numbers mentioned in the excerpt.\n\nEXCERPT (idx {idx}):\n{ex}")

answers = llm_query_batched(prompts)
for idx, ans in zip(key_idxs, answers):
    print('\n---- extracted from idx', idx, '----')
    print(ans[:1500])

SAVE(excerpts=excerpts, partial_guidelines=answers)
```

**Output:**
```

==== excerpt idx 386 chars 29390 ====
EXTERIOR WALLS 2021 INTERNATIONAL BUILDING CODE® has a wall covering of brick, concrete, stone, terra cotta, stucco or steel with minimum thicknesses in accordance with Table 1404.2. 2. Walls in which the water-resistive barrier is the only combustible component and the water-resis- tive barrier complies with the following: 2.1. A peak heat release rate of less than 150 kW/m2, a total heat release of less than 20 MJ/m2 and an effective heat of combustion of less than 18 MJ/kg when tested on spec- imens at the thickness intended for use, in accordance with ASTM E1354, in the horizontal orientation and at an incident radiant heat flux of 50 kW/m2. 2.2. A flame spread index of 25 or less and a 

==== excerpt idx 390 chars 29785 ====
EXTERIOR WALLS 2021 INTERNATIONAL BUILDING CODE® be placed between the backing and the stone veneer. The cold-formed steel framing members shall have a minimum bare steel thickness of 0.0428 inches (1.087 mm). [BS] 1404.8 Slab-type veneer. Anchored slab-type veneer units not exceeding 2 inches (51 mm) in thickness shall be anchored directly to masonry, concrete or light-frame construction. For veneer units of marble, travertine, granite or other stone units of slab form, ties of corrosion-resistant dowels in drilled holes shall be located in the middle third of the edge of the units, spaced not more than 24 inches (610 mm) apart around the periphery of each unit with not less than four ties 

==== excerpt idx 408 chars 25372 ====
ROOF ASSEMBLIES AND ROOFTOP STRUCTURES 2021 INTERNATIONAL BUILDING CODE® 1507.8.2 Deck slope. Wood shingles shall be installed on slopes of not less than three units vertical in 12 units hori- zontal (25-percent slope). 1507.8.3 Underlayment. Underlayment shall comply with Section 1507.1.1. 1507.8.4 Ice barrier. Where required, ice barriers shall comply with Section 1507.1.2. 1507.8.5 Material standards. Wood shingles shall be of naturally durable wood and comply with the require- ments of Table 1507.8.5. 1507.8.6 Attachment. Fasteners for wood shingles shall be corrosion resistant with a minimum penetration of 3/4 inch (19.1 mm) into the sheathing. For sheathing less than 1/2 inch (12.7 mm)

==== excerpt idx 491 chars 28654 ====
 SOILS AND FOUNDATIONS 2021 INTERNATIONAL BUILDING CODE® backfill height due to design earthquake ground motions. 2. The potential for liquefaction and soil strength loss evaluated for site peak ground acceleration, earthquake magnitude and source characteristics consistent with the maximum considered earth- quake ground motions. Peak ground acceleration shall be determined based on one of the following: 2.1. A site-specific study in accordance with Chapter 21 of ASCE 7. 2.2. In accordance with Section 11.8.3 of ASCE 7. 3. An assessment of potential consequences of lique- faction and soil strength loss including, but not limited to, the following: 3.1. Estimation of total and differential se

==== excerpt idx 631 chars 22179 ====
 GYPSUM BOARD, GYPSUM PANEL PRODUCTS AND PLASTER 2021 INTERNATIONAL BUILDING CODE® 2508.4 Adhesives. Gypsum board and gypsum panel products secured to framing with adhesives in ceiling assemblies shall be attached using an approved fastening schedule. Expandable foam adhesives for fastening gypsum wallboard shall conform to ASTM D6464. Other adhesives for the installation of gypsum wallboard shall conform to ASTM C557. 2508.5 Joint treatment. Gypsum board and gypsum panel product fire-resistance-rated assemblies shall have joints and fasteners treated. Exception: Joint and fastener treatment need not be provided where any of the following conditions occur: 1. Where the gypsum board or the gy

==== excerpt idx 632 chars 19064 ====
GYPSUM BOARD, GYPSUM PANEL PRODUCTS AND PLASTER 2021 INTERNATIONAL BUILDING CODE® 2508.6.1 Diaphragm proportions. The maximum allow- able diaphragm proportions shall be 11/2:1 between shear resisting elements. Rotation or cantilever conditions shall not be permitted. 2508.6.2 Installation. Gypsum board or gypsum panel products used in a horizontal diaphragm ceiling shall be installed perpendicular to ceiling framing members. End joints of adjacent courses of gypsum board shall not occur on the same joist. 2508.6.3 Blocking of perimeter edges. Perimeter edges shall be blocked using a wood member not less than 2- inch by 6-inch (51 mm by 152 mm) nominal dimension. Blocking material shall be in

==== excerpt idx 490 chars 28494 ====
 SOILS AND FOUNDATIONS 2021 INTERNATIONAL BUILDING CODE® 4. Expansion index greater than 20, determined in accordance with ASTM D4829. 1803.5.4 Ground-water table. A subsurface soil investi- gation shall be performed to determine whether the existing ground-water table is above or within 5 feet (1524 mm) below the elevation of the lowest floor level where such floor is located below the finished ground level adjacent to the foundation. Exception: A subsurface soil investigation to deter- mine the location of the gro
... [truncated]
```

---

### Iteration 9

> **Reasoning:** We have extracted partial guidelines from key IBC excerpts. Next step is to combine these partial extractions into a clean, deduplicated set of residential water-damage prevention steps, organized by construction stage/system (site, foundation, walls, roofs, wet areas), and cite the IBC section numbers and key prescriptive measurements found. Then submit final reasoning+answer.

```python
import re

partials = partial_guidelines
print('partials:', len(partials))
print('chars:', [len(p) for p in partials])

# Ask sub-LLM to synthesize into final organized guideline list with citations and measurements.
combined = "\n\n".join([f"SOURCE {i}:\n{p}" for i,p in enumerate(partials,1)])

prompt = f"""You are compiling a single, consolidated list of guidelines/steps for preventing water damage in residential construction, based strictly on the extracted IBC requirements below.

Requirements:
- Organize by topic: Site grading/drainage; Foundations/below-grade; Exterior walls & openings; Stucco/veneer details; Roof assemblies & flashings; Wet areas/interior finishes.
- Each bullet must be an actionable step/requirement.
- Preserve and include any numeric prescriptive values (slopes, distances, thicknesses, laps, clearances, drainage efficiency, membrane mil thickness, etc.).
- Include the IBC section number(s) for each bullet.
- Remove duplicates and avoid irrelevant fire/structural items.

IBC EXTRACTS:\n{combined}
"""

final = llm_query(prompt)
print(final)
SAVE(final_answer=final)
```

**Output:**
```
partials: 8
chars: [6831, 7103, 6061, 7927, 5796, 6036, 6074, 7426]
Certainly! Below is a **consolidated, organized list of actionable IBC moisture/water damage prevention guidelines and steps for residential construction**, strictly based on the provided code excerpts. Numeric prescriptive values and section references are included, duplicates removed, and grouped by topic per your instruction:

---

## 1. Site Grading and Drainage

- Grade ground adjacent to foundation to slope away at **not less than 1 unit vertical in 20 units horizontal (5% slope)** for a minimum distance of **10 feet (3048 mm)** perpendicular to foundation wall. (1804.4)  
- If 10 feet distance is not feasible, maintain a 5% slope toward an approved diversion method. (1804.4)  
- Swales within 10 feet of foundation shall slope **not less than 2%** away from building. (1804.4)  
- Impervious surfaces within 10 feet of foundation must slope **not less than 2%** away from building. (1804.4)  
- Exceptions allow slope reduction to **2% (1:48)** based on climate/soil conditions or for door landings/ramps meeting accessibility codes. (1804.4)  
- Final ground leveling must anticipate **additional settlement** of backfill. (1804.4)  
- Backfill outside foundation must be free of organic debris and compacted in lifts without damaging foundation or waterproofing. Exception: Controlled low-strength material (CLSM) need not be compacted. (1804.3)  
- Foundation drains must:  
  - Be located around foundation perimeter, consisting of gravel/crushed stone with ≤10% fines passing No. 4 sieve (4.75 mm).  
  - Extend at least **12 inches (305 mm)** beyond footing edge.  
  - Have bottom no higher than floor base bottom and top at least **6 inches (152 mm)** above footing top.  
  - Be covered with approved filter fabric membrane.  
  - Drain pipe or tile placed on **minimum 2 inches (51 mm)** gravel bedding and covered with **at least 6 inches (152 mm)** of gravel.  
  - Discharge by gravity or mechanical means into approved drainage system per International Plumbing Code.  
  - Exempted for well-drained gravel or sand/gravel soil sites. (1805.4.2, 1805.4.3)  
- Floor base course under slabs shall be minimum **4 inches (102 mm)** thick gravel/crushed stone with ≤10% fines. Exception applies for well-drained soils. (1805.4.1)  
- Finished ground level of under-floor spaces (crawl spaces) shall not be below footing bottom; groundwater control and drainage must be provided if groundwater is within **6 inches (152 mm)** of exterior grade. (1805.1.2)

---

## 2. Foundations and Below-Grade

- Dampproof walls retaining earth shall be coated on **exterior surfaces** from footing top to above finished grade with bituminous materials, acrylic-modified cement (≥3 lbs/sq yd), surface bonding mortar (1/8 inch coat min), or approved materials. (1805.2.2)  
- Surface preparation requires patching holes/recesses, parging unit masonry walls with minimum **3/8 inch** Portland cement mortar, coved at footing unless approved direct application waterproofing used. (1805.2.2.1)  
- Floors under concrete slabs require dampproofing of **not less than 6-mil (0.006 inch; 0.152 mm) polyethylene** sheeting with joints lapped **at least 6 inches (152 mm)** or equivalent approved methods. (1805.2.1)  
- Waterproofing (required where hydrostatic pressure exists and no groundwater control system) shall cover concrete/masonry walls from footing bottom to **at least 12 inches (305 mm)** above maximum groundwater elevation, with remainder dampproofed. (1805.3.2)  
- Waterproofing membranes must be rubberized asphalt, butyl rubber, fully adhered HDPE or polyolefin composite, or **minimum 6-mil (0.006 inch)** PVC with joints lapped **not less than 6 inches (152 mm)**, sealed per manufacturer. (1805.3.1, 1805.3.2)  
- Joints and penetrations in below-grade walls/floors must be sealed watertight with approved materials. (1805.3.3)  

---

## 3. Exterior Walls and Openings

- Install at least **one continuous layer of water-resistive barrier (WRB)** attached to studs or sheathing behind exterior veneer. WRB materials must comply with:  
  - No. 15 felt ASTM D226 Type 1, or  
  - ASTM E2556 Types I or II, or  
  - ASTM E331 per 1402.2, or  
  - Other approved materials per manufacturer’s instructions. (1403.2)  
- For exterior stud walls with adhered masonry veneer:  
  - Install corrosion-resistant weep screed or flashing minimum **0.019 inch (0.48 mm, No.26 gauge)** thick with vertical attachment flange minimum **3½ inches (89 mm)**. (2512.1.2, 1404.10.1.2.1)  
  - Weep screed placed not less than **4 inches (102 mm)** above finished earth or **2 inches (51 mm)** above paved surfaces. (1404.10.1.3, 2512.1.2)  
  - Water-resistive barrier must lap over screed attachment flange; exterior lath or veneer terminates on flange. (2512.1.2)  
- Flashing shall be installed at:  
  - Perimeters of doors, windows, penetrations, wall terminations, intersections with roofs, chimneys, porches, decks, 
... [truncated]
```

---

### Iteration 10

> **Reasoning:** We already have a synthesized consolidated guideline list with IBC section citations and prescriptive values saved as `final_answer`. Next step is to submit it as the final answer with brief reasoning.

```python
SUBMIT(reasoning="Extracted and consolidated actionable IBC (2021) requirements from the provided context excerpts that explicitly address moisture/water protection (grading, foundation dampproofing/waterproofing/drains, WRB/flashing, stucco drainage/weep details, roof slopes/underlayment/ice barriers/valleys/drainage, and wet-area interior material limits). Deduplicated and organized by building system, preserving section numbers and numeric criteria.", answer=final_answer)
```

**Output:**
```
SUBMIT: reasoning='Extracted and consolidated actionable IBC (2021) requirements from the provided context excerpts that explicitly address moisture/water protection (grading, foundation dampproofing/waterproofing/drains, WRB/flashing, stucco drainage/weep details, roof slopes/underlayment/ice barriers/valleys/drainage, and wet-area interior material limits). Deduplicated and organized by building system, preserving section numbers and numeric criteria.' answer='Certainly! Below is a **consolidated, organized list of actionable IBC moisture/water damage prevention guidelines and steps for residential construction**, strictly based on the provided code excerpts. Numeric prescriptive values and section references are included, duplicates removed, and grouped by topic per your instruction:\n\n---\n\n## 1. Site Grading and Drainage\n\n- Grade ground adjacent to foundation to slope away at **not less than 1 unit vertical in 20 units horizontal (5% slope)** for a minimum distance of **10 feet (3048 mm)** perpendicular to foundation wall. (1804.4)  \n- If 10 feet distance is not feasible, maintain a 5% slope toward an approved diversion method. (1804.4)  \n- Swales within 10 feet of foundation shall slope **not less than 2%** away from building. (1804.4)  \n- Impervious surfaces within 10 feet of foundation must slope **not less than 2%** away from building. (1804.4)  \n- Exceptions allow slope reduction to **2% (1:48)** based on climate/soil conditions or for door landings/ramps meeting accessibility codes. (1804.4)  \n- Final ground leveling must anticipate **additional settlement** of backfill. (1804.4)  \n- Backfill outside foundation must be free of organic debris and compacted in lifts without damaging foundation or waterproofing. Exception: Controlled low-strength material (CLSM) need not be compacted. (1804.3)  \n- Foundation drains must:  \n  - Be located around foundation perimeter, consisting of gravel/crushed stone with ≤10% fines passing No. 4 sieve (4.75 mm).  \n  - Extend at least **12 inches (305 mm)** beyond footing edge.  \n  - Have bottom no higher than floor base bottom and top at least **6 inches (152 mm)** above footing top.  \n  - Be covered with approved filter fabric membrane.  \n  - Drain pipe or tile placed on **minimum 2 inches (51 mm)** gravel bedding and covered with **at least 6 inches (152 mm)** of gravel.  \n  - Discharge by gravity or mechanical means into approved drainage system per International Plumbing Code.  \n  - Exempted for well-drained gravel or sand/gravel soil sites. (1805.4.2, 1805.4.3)  \n- Floor base course under slabs shall be minimum **4 inches (102 mm)** thick gravel/crushed stone with ≤10% fines. Exception applies for well-drained soils. (1805.4.1)  \n- Finished ground level of under-floor spaces (crawl spaces) shall not be below footing bottom; groundwater control and drainage must be provided if groundwater is within **6 inches (152 mm)** of exterior grade. (1805.1.2)\n\n---\n\n## 2. Foundations and Below-Grade\n\n- Dampproof walls retaining earth shall be coated on **exterior surfaces** from footing top to above finished grade with bituminous materials, acrylic-modified cement (≥3 lbs/sq yd), surface bonding mortar (1/8 inch coat min), or approved materials. (1805.2.2)  \n- Surface preparation requires patching holes/recesses, parging unit masonry walls with minimum **3/8 inch** Portland cement mortar, coved at footing unless approved direct application waterproofing used. (1805.2.2.1)  \n- Floors under concrete slabs require dampproofing of **not less than 6-mil (0.006 inch; 0.152 mm) polyethylene** sheeting with joints lapped **at least 6 inches (152 mm)** or equivalent approved methods. (1805.2.1)  \n- Waterproofing (required where hydrostatic pressure exists and no groundwater control system) shall cover concrete/masonry walls from footing bottom to **at least 12 inches (305 mm)** above maximum groundwater elevation, with remainder dampproofed. (1805.3.2)  \n- Waterproofing membranes must be rubberized asphalt, butyl rubber, fully adhered HDPE or polyolefin composite, or **minimum 6-mil (0.006 inch)** PVC with joints lapped **not less than 6 inches (152 mm)**, sealed per manufacturer. (1805.3.1, 1805.3.2)  \n- Joints and penetrations in below-grade walls/floors must be sealed watertight with approved materials. (1805.3.3)  \n\n---\n\n## 3. Exterior Walls and Openings\n\n- Install at least **one continuous layer of water-resistive barrier (WRB)** attached to studs or sheathing behind exterior veneer. WRB materials must comply with:  \n  - No. 15 felt ASTM D226 Type 1, or  \n  - ASTM E2556 Types I or II, or  \n  - ASTM E331 per 1402.2, or  \n  - Other approved materials per manufacturer’s instructions. (1403.2)  \n- For exterior stud walls with adhered masonry veneer:  \n  - Install corrosion-resistant weep screed or flashing minimum **0.019 inch (0.48 mm, No.26 gauge)** thick with vertical attachment flange minimum **3½ inches (89 mm)**. (2512.1.2, 1404.10
... [truncated]
```

In [20]:
Markdown(result.output.reasoning)

Extracted and consolidated actionable IBC (2021) requirements from the provided context excerpts that explicitly address moisture/water protection (grading, foundation dampproofing/waterproofing/drains, WRB/flashing, stucco drainage/weep details, roof slopes/underlayment/ice barriers/valleys/drainage, and wet-area interior material limits). Deduplicated and organized by building system, preserving section numbers and numeric criteria.

In [21]:
Markdown(result.output.answer)

Certainly! Below is a **consolidated, organized list of actionable IBC moisture/water damage prevention guidelines and steps for residential construction**, strictly based on the provided code excerpts. Numeric prescriptive values and section references are included, duplicates removed, and grouped by topic per your instruction:

---

## 1. Site Grading and Drainage

- Grade ground adjacent to foundation to slope away at **not less than 1 unit vertical in 20 units horizontal (5% slope)** for a minimum distance of **10 feet (3048 mm)** perpendicular to foundation wall. (1804.4)  
- If 10 feet distance is not feasible, maintain a 5% slope toward an approved diversion method. (1804.4)  
- Swales within 10 feet of foundation shall slope **not less than 2%** away from building. (1804.4)  
- Impervious surfaces within 10 feet of foundation must slope **not less than 2%** away from building. (1804.4)  
- Exceptions allow slope reduction to **2% (1:48)** based on climate/soil conditions or for door landings/ramps meeting accessibility codes. (1804.4)  
- Final ground leveling must anticipate **additional settlement** of backfill. (1804.4)  
- Backfill outside foundation must be free of organic debris and compacted in lifts without damaging foundation or waterproofing. Exception: Controlled low-strength material (CLSM) need not be compacted. (1804.3)  
- Foundation drains must:  
  - Be located around foundation perimeter, consisting of gravel/crushed stone with ≤10% fines passing No. 4 sieve (4.75 mm).  
  - Extend at least **12 inches (305 mm)** beyond footing edge.  
  - Have bottom no higher than floor base bottom and top at least **6 inches (152 mm)** above footing top.  
  - Be covered with approved filter fabric membrane.  
  - Drain pipe or tile placed on **minimum 2 inches (51 mm)** gravel bedding and covered with **at least 6 inches (152 mm)** of gravel.  
  - Discharge by gravity or mechanical means into approved drainage system per International Plumbing Code.  
  - Exempted for well-drained gravel or sand/gravel soil sites. (1805.4.2, 1805.4.3)  
- Floor base course under slabs shall be minimum **4 inches (102 mm)** thick gravel/crushed stone with ≤10% fines. Exception applies for well-drained soils. (1805.4.1)  
- Finished ground level of under-floor spaces (crawl spaces) shall not be below footing bottom; groundwater control and drainage must be provided if groundwater is within **6 inches (152 mm)** of exterior grade. (1805.1.2)

---

## 2. Foundations and Below-Grade

- Dampproof walls retaining earth shall be coated on **exterior surfaces** from footing top to above finished grade with bituminous materials, acrylic-modified cement (≥3 lbs/sq yd), surface bonding mortar (1/8 inch coat min), or approved materials. (1805.2.2)  
- Surface preparation requires patching holes/recesses, parging unit masonry walls with minimum **3/8 inch** Portland cement mortar, coved at footing unless approved direct application waterproofing used. (1805.2.2.1)  
- Floors under concrete slabs require dampproofing of **not less than 6-mil (0.006 inch; 0.152 mm) polyethylene** sheeting with joints lapped **at least 6 inches (152 mm)** or equivalent approved methods. (1805.2.1)  
- Waterproofing (required where hydrostatic pressure exists and no groundwater control system) shall cover concrete/masonry walls from footing bottom to **at least 12 inches (305 mm)** above maximum groundwater elevation, with remainder dampproofed. (1805.3.2)  
- Waterproofing membranes must be rubberized asphalt, butyl rubber, fully adhered HDPE or polyolefin composite, or **minimum 6-mil (0.006 inch)** PVC with joints lapped **not less than 6 inches (152 mm)**, sealed per manufacturer. (1805.3.1, 1805.3.2)  
- Joints and penetrations in below-grade walls/floors must be sealed watertight with approved materials. (1805.3.3)  

---

## 3. Exterior Walls and Openings

- Install at least **one continuous layer of water-resistive barrier (WRB)** attached to studs or sheathing behind exterior veneer. WRB materials must comply with:  
  - No. 15 felt ASTM D226 Type 1, or  
  - ASTM E2556 Types I or II, or  
  - ASTM E331 per 1402.2, or  
  - Other approved materials per manufacturer’s instructions. (1403.2)  
- For exterior stud walls with adhered masonry veneer:  
  - Install corrosion-resistant weep screed or flashing minimum **0.019 inch (0.48 mm, No.26 gauge)** thick with vertical attachment flange minimum **3½ inches (89 mm)**. (2512.1.2, 1404.10.1.2.1)  
  - Weep screed placed not less than **4 inches (102 mm)** above finished earth or **2 inches (51 mm)** above paved surfaces. (1404.10.1.3, 2512.1.2)  
  - Water-resistive barrier must lap over screed attachment flange; exterior lath or veneer terminates on flange. (2512.1.2)  
- Flashing shall be installed at:  
  - Perimeters of doors, windows, penetrations, wall terminations, intersections with roofs, chimneys, porches, decks, balconies, built-in gutters, and similar locations susceptible to moisture intrusion. (1404.4)  
  - Flashing must prevent moisture ingress or redirect water to exterior or WRB as part of drainage plane per 1402.2.  
  - Flashing metal thickness minimum **0.019 inch (0.48 mm, No. 26 gauge)**, corrosion resistant. (1404.4, 1503.2.1)  
  - Flashings with projecting flanges are required on both sides and ends of copings, under window sills, and continuously above projecting trim. (1404.4)  
  - Avoid wall pockets or crevices; protect if unavoidable with capping or drips. (1404.4.1)  
- Flashing and weep holes for anchored veneer shall be located in first course above shelf angles, lintels, or structural supports, with maximum 10 inches (245 mm) above finished grade. (1404.4.2)  
- Use corrosion-resistant fasteners for siding and veneers: aluminum, copper, zinc, galvanized steel, or other approved materials. (1404.14, 1404.17)  
- Fastening requirements for vinyl siding: nails corrosion resistant, head diameter ≥ 0.313 inch (7.9 mm), shank diameter ≥ 1/8 inch (3.18 mm), penetration into substrate ≥ 1¼ inch (32 mm), spacing ≤ 16 inches (406 mm) for horizontal siding; ≤ 12 inches (305 mm) for vertical siding. (1404.14.1.1, 1404.14.1.3)  
- Fastening for fiber-cement siding: corrosion-resistant nails, round head, smooth shank, penetrating studs ≥ 1 inch (25 mm); steel framing use screws penetrating at least 3 exposed threads. (1404.16)  
- Wood veneers minimum nominal thickness: **1 inch (25 mm)** or 0.438 inch (11.1 mm) exterior hardboard siding or 0.375 inch (9.5 mm) exterior structural panels. (1404.5)  
- Minimum siding thicknesses for weather protection examples:  
  - Fiber cement siding: **0.25 inch**  
  - Vinyl siding: **0.035 inch**  
  - Stucco or cement plaster (3-coat over metal lath): **0.875 inch**  
  - Aluminum siding: **0.019 inch** (1404.2)  

---

## 4. Stucco / Veneer Details

- For adhered masonry veneer with metal lath (stucco):  
  - Water-resistive barriers must be installed per 1403.2 and 2510.6 (see below for climate-specific requirements). (2510.6, 1403.2)  
  - In dry climates (Climate Zone B):  
    - Either two layers of 10-minute Grade D paper or ASTM E2556 Type I WRB as separate planes with flashing drainage between layers, or  
    - One layer of 60-minute Grade D paper or ASTM E2556 Type II WRB separated from stucco by foam plastic sheathing, nonwater absorbing layer, or drainage space. (2510.6.1)  
  - In moist or marine climates (Climate Zone A or C):  
    - Besides above, provide drainage space minimum **3/16 inch (4.8 mm)** depth or drainage material with minimum **90% drainage efficiency** per ASTM E2273 or ASTM E2925 on exterior side of WRB. (2510.6.2)  
- Masonry/concrete surfaces prepared for plaster/stucco must be clean, free of efflorescence, damp, and rough — or receive approved bonding agents or a Portland cement dash bond coat mixed max **2 parts sand to 1 part cement** by volume and moist-cured minimum **24 hours**. (2510.7)  
- Apply lathing as per ASTM C1063; nominal **½ inch (12.7 mm)** mortar scratch coat over lath, scored horizontally; base coat setting bed mortar nominal **½ inch (12.7 mm)** thick before veneer placement, resulting in net **3/8 inch (9.5 mm)** thick setting bed after placement. (1404.10.1.4)  
- Weep screeds for stucco installed at or below foundation plate line must be corrosion resistant, minimum **0.019 inch (0.48 mm)** thick, with **3½ inch (89 mm)** vertical flange, located not less than **4 inches (102 mm)** above earth or **2 inches (51 mm)** above paved surfaces; the WRB must lap over flange. Exterior lath terminates on flange. (2512.1.2, 1404.10.1.2.1)  
- Wood framing behind plaster or tile in wet/splash areas (showers, wet walls) must be protected by an approved moisture barrier. (2511.5)  
- Gypsum board or lath backing is not permitted for plaster unless with WRB and only on:  
  - Horizontal ceilings/roof soffits with self-furred metal lath, or  
  - Interior walls.  
- Gypsum sheathing with WRB is permitted as backing for plaster on walls. (2510.5)  
- Stucco and plaster materials must be stored protected from weather prior to installation. (2510.2)  

---

## 5. Roof Assemblies and Flashings

- Wood shingles: installed on slopes **not less than 3 units vertical in 12 units horizontal (25%)** slope. (1507.8.2)  
- Wood shakes: installed on slopes **not less than 4 units vertical in 12 units horizontal (33%)**. (1507.9.2)  
- Built-up, modified bitumen, single-ply, sprayed polyurethane foam, and liquid-applied roofs require minimum slopes of **1/4 unit vertical in 12 units horizontal (2%)** except coal-tar built-up roofs which require minimum slope of **1/8 unit vertical in 12 (1%)**. (1507.10.1 – 1507.14.1)  
- Photovoltaic shingles and BIPV roof panels require minimum slope **2 units vertical in 12 units horizontal (2:12)**. (1507.16.2, 1507.17.2)  
- Wood shingles/shakes applied to solid or spaced sheathing; in cold climates (<25°F January avg) or ice-backup areas, solid sheathing mandatory. (1507.8, 1507.9)  
- Roof underlayment must comply with Section 1507.1.1 and ASTM D226 (Type I for wood shakes), applied shingle fashion starting at eave with minimum **2-inch (50 mm)** laps. (1507.8.3, 1507.9.5, 1507.17.4)  
- Ice barriers (where required) must have at least two waterproof underlayment layers extending from lowest edge of roof to at least **24 inches (610 mm)** inside exterior wall line. (1507.8.4, 1507.9.4, 1507.17.4.2)  
- Flashings at roof-to-wall intersections, penetrations, gutters, and roof slope changes: corrosion-resistant metal minimum thickness **0.019 inch (0.48 mm, No. 26 gauge)**. (1503.2, 1503.2.1)  
- Valley flashings: minimum extension **11 inches (280 mm)** each side of centerline with a minimum **1 inch (25 mm)** high splash diverter rib; 4-inch end laps minimum. (1507.8.8)  
- For slopes ≥ 3:12, valley flashing underlayment requires Type I underlayment or self-adhered polymer modified bitumen sheet (ASTM D1970) in addition to main underlayment. (1507.8.8)  
- Fasteners for wood shingles/shakes: corrosion resistant (hot-dip galvanized or stainless steel), minimum penetration into sheathing **3/4 inch (19.1 mm)**; if sheathing <1/2 inch thick, fasteners penetrate through sheathing; two fasteners per shingle/shake minimum. (1507.8.6, 1507.9.7)  
- Fasteners for photovoltaic shingles: galvanized, stainless steel, aluminum or copper roofing nails; minimum 12-gage shank (0.105 inch diameter), minimum 3/8 inch head diameter, minimum 3/4 inch penetration; full penetration if sheathing <3/4 inch thick. (1507.16.5)  
- Roof drainage systems must include primary and secondary (emergency overflow) drainage sized to prevent ponding; scuppers required where roof parapets create water pockets must have openings **not less than 4 inches (102 mm)**. (1502.2, 1502.3)  
- Gutters, downspouts: constructed of noncombustible materials or Schedule 40 plastic (except some residential exceptions). (1502.4)  
- Parapet walls must be coped or covered with weatherproof materials providing **positive drainage**, with covering width **not less than thickness of parapet wall**. (1503.3, 1503.3.1, 1503.3.2)  
- Install crickets or saddles on ridge side of chimneys or penetrations wider than **30 inches (762 mm)** to divert water away. (1503.5)  
- Attic and rafter ventilation must comply with Section 1202.2 and manufacturer instructions to manage moisture. (1503.4)  

---

## 6. Wet Areas and Interior Finish Systems

- For **showers and water closets**:  
  - Wall tile/ceiling panel bases must be materials listed in Table 2509.2 and installed per manufacturer instructions. (2509.2)  
  - Use **water-resistant gypsum backing board** for tile bases in water closet compartments, installed per GA 216 or ASTM C840 and manufacturer instructions. (2509.2)  
  - Regular gypsum wallboard permitted under tile/wall panels elsewhere if installed per GA 216 or ASTM C840. (2509.2)  
  - Water-resistant gypsum backing boards must NOT be installed over vapor retarders or where continuous water exposure/high humidity exists. (2509.3)  
- Wood framing in wet or splash areas (e.g., showers) covered with plaster or tile must be protected with an approved moisture barrier. (2511.5)  

---

**End of consolidated list.**

---

Please advise if you want this formatted as a checklist, or focused on specific materials/assemblies.

In [17]:
result.usage.__dict__

{'input_tokens': 190799,
 'cache_write_tokens': 0,
 'cache_read_tokens': 152832,
 'output_tokens': 3023,
 'input_audio_tokens': 0,
 'cache_audio_read_tokens': 0,
 'output_audio_tokens': 0,
 'details': {'accepted_prediction_tokens': 0,
  'audio_tokens': 0,
  'reasoning_tokens': 0,
  'rejected_prediction_tokens': 0},
 'requests': 9,
 'tool_calls': 0}